# Гатчина: индикаторы кварталов и интерактивный дашборд

Ноутбук выполняет весь путь последовательно, **без остановок на ручную проверку данных и без согласования метрик**:

1. читает готовый файл `examples/data/blocks_agg_with_indicators.geojson`;
2. проверяет геометрию, пропуски, диапазоны и согласованность показателей;
3. объясняет, что означает каждый показатель и в каких единицах он измеряется;
4. пересчитывает площади и производные индикаторы;
5. собирает сводные таблицы, графики и выводы;
6. сохраняет все результаты;
7. создаёт **одну HTML-страницу** с интерактивной картой и дашбордом.

**Город:** Гатчина, Ленинградская область.  
**Источник данных:** агрегированные индикаторы кварталов BlocksNet / OpenStreetMap.  
**Ориентир по методике:** [examples/pipeline.ipynb BlocksNet](https://github.com/aimclub/blocksnet/blob/main/examples/pipeline.ipynb).

> В исходном файле уже есть результаты агрегирования зданий, землепользования, плотности и доступности. Поэтому шаги скачивания OSM и нарезки кварталов не повторяются. Школы и обеспеченность не рассчитываются: соответствующих полей в файле нет.


## Как запустить

Из корня проекта:

```powershell
cd "C:\Users\pc\Desktop\р\python-package-boilerplate"
.\.venv\Scripts\Activate.ps1
jupyter lab examples\gatchina_dashboard.ipynb
```

Затем: **Kernel → Restart & Run All**.

Или полностью из командной строки:

```powershell
.\.venv\Scripts\jupyter.exe nbconvert --to notebook --execute --inplace examples\gatchina_dashboard.ipynb
```

После выполнения откройте:

```text
examples\gatchina_dashboard.html
```

Страница самодостаточна: библиотеки Leaflet и Chart.js встроены в HTML, данные тоже. Подложка OpenStreetMap подгружается из интернета; кварталы, графики и таблицы работают и без сети.


## Что означают показатели

Сначала — смысл и единицы, потом — расчёты. Так таблицы и карта читаются без специальных знаний.

| Показатель | Единица | Что означает | Что помогает понять |
|---|---|---|---|
| `population` | человек | Оценка населения квартала, полученная агрегированием зданий и нормативной жилплощади | Где сосредоточены жители |
| `population_density` | чел./га | Население на гектар территории квартала | Насколько плотно заселена территория |
| `site_area` | м² | Площадь квартала **в исходном файле** | Для сверки; в этом файле она посчитана в EPSG:3857 и завышена |
| `area_m2` | м² | Площадь квартала в метрической проекции UTM 36N | Честная физическая площадь |
| `footprint_area` | м² | Суммарная площадь застройки в плане | Сколько территории занято зданиями |
| `build_floor_area` | м² | Суммарная поэтажная площадь зданий | Объём застройки |
| `living_area` | м² | Жилая площадь | Жилищный фонд |
| `non_living_area` | м² | Нежилая площадь | Деловая, сервисная и иная нагрузка |
| `fsi` | доля | Floor Space Index: `build_floor_area / site_area` в исходном файле | Интенсивность застройки, но с завышенным знаменателем |
| `fsi_calc` | доля | FSI, пересчитанный на честную `area_m2` | Настоящая интенсивность застройки |
| `gsi` | доля | Ground Space Index: `footprint_area / site_area` в исходном файле | Плотность застройки в плане, но с завышенным знаменателем |
| `gsi_calc` | доля | GSI, пересчитанный на честную `area_m2` | Настоящая доля застроенной территории |
| `mxi` | доля | MXI: доля жилой площади в общей поэтажной площади | Насколько квартал жилой |
| `l` | этажей | Средняя этажность: `build_floor_area / footprint_area` | Высотность застройки |
| `area_accessibility` | минуты | Оценка доступности территории | Насколько квартал удалён от основных точек притяжения |
| `land_use` | категория | Преобладающее землепользование квартала | Функциональная структура города |
| `morphotype` | категория | Морфотип Strelka, рассчитанный по плотностным индикаторам | Тип застройки: индивидуальная, малоэтажная, среднеэтажная |
| `living_per_capita` | м²/чел. | Жилая площадь на человека | Комфорт жилищного обеспечения |
| `floor_per_capita` | м²/чел. | Общая поэтажная площадь на человека | Общая обеспеченность площадью |

**Важная поправка.** В исходном файле `site_area` совпадает с площадью в EPSG:3857 (Web Mercator), а не с метрической площадью. Поэтому исходные `fsi` и `gsi` занижены. В выводах используем `fsi_calc` и `gsi_calc`, а исходные значения показываем рядом для прозрачности.


In [1]:
from __future__ import annotations

import html
import json
import math
import re
import sys
import urllib.request
from datetime import date
from importlib.metadata import version
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import mapping
from shapely.ops import unary_union
from shapely.validation import make_valid

from blocksnet.analysis.morphotypes import get_strelka_morphotypes

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)


def find_project_root() -> Path:
    """Находит корень проекта независимо от того, откуда запущен ноутбук."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "examples" / "data" / "blocks_agg_with_indicators.geojson").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден examples/data/blocks_agg_with_indicators.geojson. "
        "Запустите ноутбук из корня проекта или из папки examples."
    )


ROOT = find_project_root()
EXAMPLES = ROOT / "examples"
SOURCE = EXAMPLES / "data" / "blocks_agg_with_indicators.geojson"
OUT_DATA = EXAMPLES / "data" / "gatchina"
VENDOR = OUT_DATA / "vendor"
HTML_PATH = EXAMPLES / "gatchina_dashboard.html"
OUT_DATA.mkdir(parents=True, exist_ok=True)

CITY = "Гатчина"
REGION = "Ленинградская область"
SOURCE_NOTE = "Источник: агрегированные индикаторы кварталов BlocksNet / OpenStreetMap."

print("Город:", CITY, "—", REGION)
print("Данные:", SOURCE.resolve())
print("Результаты:", OUT_DATA.resolve())
print("HTML:", HTML_PATH.resolve())
print("Python:", sys.version.split()[0])
print("GeoPandas:", gpd.__version__)
print("Pandas:", pd.__version__)
print("BlocksNet:", version("blocksnet"))


Город: Гатчина — Ленинградская область
Данные: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\blocks_agg_with_indicators.geojson
Результаты: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\gatchina
HTML: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\gatchina_dashboard.html
Python: 3.12.10
GeoPandas: 1.2.0
Pandas: 2.3.3
BlocksNet: 1.0.0a9


In [2]:
if not SOURCE.exists():
    raise FileNotFoundError(f"Нет файла данных: {SOURCE}")

gdf = gpd.read_file(SOURCE)

print("Объектов:", len(gdf))
print("CRS:", gdf.crs)
print("Типы геометрии:", gdf.geom_type.value_counts().to_dict())
print("Колонки:", ", ".join(map(str, gdf.columns)))
display(gdf.head(3))


Объектов: 351
CRS: EPSG:4326
Типы геометрии: {'Polygon': 351}
Колонки: residential, business, recreation, industrial, transport, special, agriculture, land_use, share, footprint_area, build_floor_area, living_area, non_living_area, population, site_area, fsi, gsi, mxi, l, morphotype, area_accessibility, geometry


,residential,business,recreation,industrial,transport,special,agriculture,land_use,share,footprint_area,build_floor_area,living_area,non_living_area,population,site_area,fsi,gsi,mxi,l,morphotype,area_accessibility,geometry
0,0.690207,0.211718,0.055976,0.0,0.042093,0.0,0.0,LandUse.RESIDENTIAL,0.690207,7383.849525,10126.647386,930.718518,9195.928868,30.0,836277.821580,0.012109,0.008829,0.091908,1.371459,individual residential,10.063987,"POLYGON ((30.09367 59.55207, 30.09375 59.55215..."
1,0.970519,0.029484,0.000000,0.0,0.000000,0.0,0.0,LandUse.RESIDENTIAL,0.970519,15304.958732,89136.104333,63508.757346,25627.346987,2000.0,420144.041613,0.212156,0.036428,0.712492,5.824002,mid-rise,10.210398,"POLYGON ((30.09448 59.56164, 30.09448 59.56164..."
2,0.936589,0.010562,0.052849,0.0,0.000000,0.0,0.0,LandUse.RESIDENTIAL,0.936589,17141.628200,91971.715857,72292.449720,19679.266138,2275.0,302985.560227,0.303551,0.056576,0.786029,5.365401,mid-rise,8.669212,"POLYGON ((30.12546 59.58002, 30.12541 59.57998..."


## Проверки данных

Проверки выполняются автоматически. Ноутбук не останавливается на «посмотреть глазами»: все предупреждения собираются в отчёт, а критические ошибки сразу прерывают выполнение. Предупреждения не мешают строить дашборд, но попадают в раздел «Методика и проверки».


In [3]:
checks = []


def add_check(name, status, detail):
    checks.append({"Проверка": name, "Статус": status, "Комментарий": str(detail)})


required_columns = [
    "residential", "business", "recreation", "industrial", "transport",
    "special", "agriculture", "land_use", "share", "footprint_area",
    "build_floor_area", "living_area", "non_living_area", "population",
    "site_area", "fsi", "gsi", "mxi", "l", "morphotype", "area_accessibility",
]
missing_columns = [c for c in required_columns if c not in gdf.columns]
add_check(
    "Обязательные колонки",
    "ERROR" if missing_columns else "OK",
    "нет: " + ", ".join(missing_columns) if missing_columns else "все обязательные колонки на месте",
)
if missing_columns:
    raise RuntimeError("В файле нет обязательных колонок: " + ", ".join(missing_columns))

add_check("Число объектов", "OK" if len(gdf) > 0 else "ERROR", f"{len(gdf)} кварталов")
add_check("CRS исходного файла", "OK" if gdf.crs is not None else "WARN", str(gdf.crs))

geom_types = gdf.geom_type.value_counts().to_dict()
bad_types = [k for k in geom_types if k not in ("Polygon", "MultiPolygon")]
add_check(
    "Типы геометрии",
    "WARN" if bad_types else "OK",
    "; ".join(f"{k}: {v}" for k, v in geom_types.items()),
)

invalid_count = int((~gdf.geometry.is_valid).sum())
add_check(
    "Валидность геометрии",
    "WARN" if invalid_count else "OK",
    f"невалидных: {invalid_count} — будут исправлены через make_valid",
)

empty_count = int(gdf.geometry.isna().sum() + gdf.geometry.is_empty.sum())
add_check(
    "Пустая геометрия",
    "WARN" if empty_count else "OK",
    f"пустых/None: {empty_count}",
)

duplicate_geoms = int(gdf.geometry.apply(lambda g: g.wkb if g is not None else None).duplicated().sum())
add_check(
    "Дубли геометрии",
    "WARN" if duplicate_geoms else "OK",
    f"полных дублей: {duplicate_geoms}",
)

numeric_cols = [
    "residential", "business", "recreation", "industrial", "transport",
    "special", "agriculture", "share", "footprint_area", "build_floor_area",
    "living_area", "non_living_area", "population", "site_area",
    "fsi", "gsi", "mxi", "l", "area_accessibility",
]
for col in numeric_cols:
    gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

null_counts = {col: int(n) for col, n in gdf.isna().sum().items() if int(n) > 0}
add_check(
    "Пропуски",
    "WARN" if null_counts else "OK",
    ", ".join(f"{k}: {v}" for k, v in null_counts.items()) if null_counts else "пропусков нет",
)

range_issues = []
for col in ["residential", "business", "recreation", "industrial", "transport", "special", "agriculture", "share"]:
    bad = int(((gdf[col] < -1e-3) | (gdf[col] > 1 + 1e-3)).sum())
    if bad:
        range_issues.append(f"{col}: {bad} вне [0, 1] (допуск 0.001)")
for col in ["footprint_area", "build_floor_area", "living_area", "non_living_area", "population", "site_area", "area_accessibility"]:
    bad = int((gdf[col] < 0).sum())
    if bad:
        range_issues.append(f"{col}: {bad} отрицательных")
add_check(
    "Диапазоны значений",
    "WARN" if range_issues else "OK",
    "; ".join(range_issues) if range_issues else "отрицательных площадей и долей вне [0, 1] нет",
)

land_use_values = sorted(gdf["land_use"].dropna().astype(str).unique())
allowed_land_use = {
    "LandUse.RESIDENTIAL", "LandUse.BUSINESS", "LandUse.RECREATION",
    "LandUse.INDUSTRIAL", "LandUse.TRANSPORT", "LandUse.SPECIAL",
    "LandUse.AGRICULTURE",
}
unknown_land_use = [v for v in land_use_values if v not in allowed_land_use]
add_check(
    "Категории землепользования",
    "WARN" if unknown_land_use else "OK",
    "неизвестные: " + ", ".join(unknown_land_use) if unknown_land_use else f"{len(land_use_values)} ожидаемых категорий",
)

morph_values = sorted(gdf["morphotype"].dropna().astype(str).unique())
allowed_morph = {
    "individual residential", "low-rise non-residential", "low-rise model",
    "mid-rise", "mid-rise microdistrict",
}
unknown_morph = [v for v in morph_values if v not in allowed_morph]
add_check(
    "Категории морфотипов",
    "WARN" if unknown_morph else "OK",
    "неизвестные: " + ", ".join(unknown_morph) if unknown_morph else f"{len(morph_values)} ожидаемых категорий",
)


def median_rel_error(actual, expected):
    a = pd.to_numeric(pd.Series(actual), errors="coerce")
    e = pd.to_numeric(pd.Series(expected), errors="coerce")
    mask = a.notna() & e.notna() & (e.abs() > 0)
    if not mask.any():
        return float("nan")
    return float(((a[mask] - e[mask]).abs() / e[mask].abs()).median())


formula_specs = {
    "FSI: файл = build_floor_area / site_area": (gdf["fsi"], gdf["build_floor_area"] / gdf["site_area"].replace(0, np.nan)),
    "GSI: файл = footprint_area / site_area": (gdf["gsi"], gdf["footprint_area"] / gdf["site_area"].replace(0, np.nan)),
    "MXI = living_area / build_floor_area": (gdf["mxi"], gdf["living_area"] / gdf["build_floor_area"].replace(0, np.nan)),
    "L = build_floor_area / footprint_area": (gdf["l"], gdf["build_floor_area"] / gdf["footprint_area"].replace(0, np.nan)),
}
for name, (actual, expected) in formula_specs.items():
    err = median_rel_error(actual, expected)
    add_check(
        name,
        "OK" if (pd.notna(err) and err < 0.02) else "WARN",
        f"медианная относительная ошибка: {err:.4f}" if pd.notna(err) else "нет данных для проверки",
    )

pop_expected = gdf["living_area"] / 30.0
pop_err = median_rel_error(gdf.loc[gdf["population"] > 0, "population"], pop_expected[gdf["population"] > 0])
add_check(
    "Население ≈ жилплощадь / 30",
    "OK" if (pd.notna(pop_err) and pop_err < 0.15) else "WARN",
    f"медианная относительная ошибка: {pop_err:.4f}" if pd.notna(pop_err) else "нет жилых кварталов",
)

share_sum = gdf[["residential", "business", "recreation", "industrial", "transport", "special", "agriculture"]].sum(axis=1, min_count=1)
share_bad = int(((share_sum < 0.98) | (share_sum > 1.02)).sum())
add_check(
    "Сумма долей землепользования",
    "WARN" if share_bad else "OK",
    f"строк с суммой вне [0.98, 1.02]: {share_bad}; сумма может быть меньше 1, если часть квартала не классифицирована",
)

max_share = gdf[["residential", "business", "recreation", "industrial", "transport", "special", "agriculture"]].max(axis=1)
share_label_err = median_rel_error(gdf["share"], max_share)
add_check(
    "share = максимальная доля",
    "OK" if (pd.notna(share_label_err) and share_label_err < 0.02) else "WARN",
    f"медианная относительная ошибка: {share_label_err:.4f}" if pd.notna(share_label_err) else "нет данных",
)

area_3857 = gdf.to_crs(3857).area
area_ratio_3857 = gdf["site_area"] / area_3857.replace(0, np.nan)
ratio_median = float(area_ratio_3857.median())
add_check(
    "site_area и EPSG:3857",
    "WARN" if abs(ratio_median - 1.0) < 1e-6 else "OK",
    f"медиана site_area / площадь(3857) = {ratio_median:.6f}; 1.0 означает Web Mercator, а не метрическую площадь",
)

utm_crs = gdf.estimate_utm_crs()
area_utm = gdf.to_crs(utm_crs).area
area_ratio_utm = gdf["site_area"] / area_utm.replace(0, np.nan)
add_check(
    "site_area и UTM",
    "OK",
    f"медиана site_area / площадь({utm_crs}) = {float(area_ratio_utm.median()):.4f}; это показывает масштаб завышения",
)

validation_df = pd.DataFrame(checks)
display(validation_df)

if (validation_df["Статус"] == "ERROR").any():
    bad = validation_df.loc[validation_df["Статус"] == "ERROR", "Проверка"].tolist()
    raise RuntimeError("Критические ошибки данных: " + ", ".join(bad))


,Проверка,Статус,Комментарий
0,Обязательные колонки,OK,все обязательные колонки на месте
1,Число объектов,OK,351 кварталов
2,CRS исходного файла,OK,EPSG:4326
3,Типы геометрии,OK,Polygon: 351
4,Валидность геометрии,WARN,невалидных: 6 — будут исправлены через make_valid
5,Пустая геометрия,OK,пустых/None: 0
6,Дубли геометрии,OK,полных дублей: 0
7,Пропуски,WARN,"land_use: 7, share: 7, gsi: 2, morphotype: 55"
8,Диапазоны значений,OK,"отрицательных площадей и долей вне [0, 1] нет"
9,Категории землепользования,OK,6 ожидаемых категорий


## Очистка и производные показатели

Что делаем и зачем:

- **Исправляем геометрию** через `make_valid`: 6 невалидных полигонов иначе дадут ошибки при расчёте площади и отрисовке. У двух из них после исправления не остаётся площади (они схлопываются в линии), поэтому в итоговой выборке 349 кварталов из 351; это явно фиксируется в отчёте проверок.
- **Считаем площадь в UTM 36N** (`area_m2`): это метрическая площадь, а не Web Mercator. Для Гатчины подходит UTM 36N.
- **Пересчитываем FSI/GSI** на честную площадь: `fsi_calc = build_floor_area / area_m2`, `gsi_calc = footprint_area / area_m2`.
- **Добавляем производные показатели**: плотность населения, жилплощадь на человека, доля нежилой площади, класс доступности.
- **Переводим категории на русский** и сохраняем исходные значения в отдельных колонках.
- **Пересчитываем морфотипы Strelka** по исправленному FSI/GSI и сравниваем с исходной колонкой `morphotype`. Это не замена исходных данных, а проверка чувствительности классификации к площади.


In [4]:
gdf = gdf.copy()
blocks_before_cleaning = len(gdf)
invalid_before = int((~gdf.geometry.is_valid).sum())


def polygonal(geom):
    """Оставляет только полигональную часть геометрии."""
    if geom is None or geom.is_empty:
        return geom
    if geom.geom_type in ("Polygon", "MultiPolygon"):
        return geom
    parts = [part for part in getattr(geom, "geoms", []) if part.geom_type in ("Polygon", "MultiPolygon")]
    return unary_union(parts) if parts else geom


gdf["geometry"] = gdf.geometry.apply(lambda g: polygonal(make_valid(g)) if g is not None and not g.is_empty else g)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf = gdf.reset_index(drop=True)
blocks_after_cleaning = len(gdf)
dropped_zero_area = blocks_before_cleaning - blocks_after_cleaning
gdf["block_id"] = np.arange(len(gdf), dtype=int)

UTM_CRS = gdf.estimate_utm_crs()
gdf["area_m2"] = gdf.to_crs(UTM_CRS).area.values
gdf["area_ha"] = gdf["area_m2"] / 10000.0
gdf["source_area_km2"] = pd.to_numeric(gdf["site_area"], errors="coerce") / 1e6
gdf["site_area_ratio"] = gdf["site_area"] / gdf["area_m2"].replace(0, np.nan)
gdf["area_3857_m2"] = gdf.to_crs(3857).area.values
gdf["site_area_3857_ratio"] = gdf["site_area"] / pd.Series(gdf["area_3857_m2"]).replace(0, np.nan)

LAND_USE_MAP = {
    "residential": "Жилая",
    "business": "Деловая",
    "recreation": "Рекреационная",
    "industrial": "Промышленная",
    "transport": "Транспортная",
    "special": "Специальная",
    "agriculture": "Сельскохозяйственная",
}
MORPH_MAP = {
    "individual residential": "Индивидуальная жилая застройка",
    "low-rise model": "Малоэтажная модель",
    "low-rise non-residential": "Малоэтажная нежилая",
    "mid-rise": "Среднеэтажная",
    "mid-rise microdistrict": "Среднеэтажный микрорайон",
}


def normalize_land_use(value):
    if pd.isna(value):
        return "Не определено"
    text = str(value).replace("LandUse.", "").strip().lower()
    return LAND_USE_MAP.get(text, "Другое")


def normalize_morph(value):
    if pd.isna(value):
        return "Не определено"
    text = str(value).strip().lower()
    return MORPH_MAP.get(text, "Другое")


gdf["land_use_ru"] = gdf["land_use"].apply(normalize_land_use)
gdf["morphotype_ru"] = gdf["morphotype"].apply(normalize_morph)


def safe_div(numerator, denominator):
    a = pd.to_numeric(pd.Series(numerator), errors="coerce")
    b = pd.to_numeric(pd.Series(denominator), errors="coerce")
    result = a / b.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)


gdf["population_density"] = safe_div(gdf["population"], gdf["area_m2"] / 10000.0).values
gdf["living_per_capita"] = safe_div(gdf["living_area"], gdf["population"]).values
gdf["floor_per_capita"] = safe_div(gdf["build_floor_area"], gdf["population"]).values
gdf["non_living_share"] = safe_div(gdf["non_living_area"], gdf["build_floor_area"]).values
gdf["fsi_calc"] = safe_div(gdf["build_floor_area"], gdf["area_m2"]).values
gdf["gsi_calc"] = safe_div(gdf["footprint_area"], gdf["area_m2"]).values
gdf["land_use_share_total"] = gdf[["residential", "business", "recreation", "industrial", "transport", "special", "agriculture"]].sum(axis=1, min_count=1)
gdf["is_developed"] = gdf["build_floor_area"].fillna(0) > 0

access_class = pd.cut(
    gdf["area_accessibility"],
    bins=[-np.inf, 10, 15, 20, np.inf],
    labels=["до 10 мин", "10–15 мин", "15–20 мин", "более 20 мин"],
    right=True,
)
gdf["access_class"] = access_class.astype(object).where(access_class.notna(), "Нет данных")

strelka_input = pd.DataFrame(
    {
        "fsi": gdf["fsi_calc"],
        "gsi": gdf["gsi_calc"],
        "mxi": gdf["mxi"],
        "l": gdf["l"],
    },
    index=gdf.index,
)
strelka_raw = get_strelka_morphotypes(strelka_input)["morphotype"]
gdf["morphotype_recalc"] = strelka_raw.apply(normalize_morph)
gdf["morphotype_changed"] = gdf["morphotype_ru"] != gdf["morphotype_recalc"]


def json_safe(value):
    """Приводит значение к виду, который безопасно сериализуется в JSON."""
    if value is None:
        return None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (int, float)):
        return None if isinstance(value, float) and not math.isfinite(value) else value
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, (str, bytes)):
        return value.decode("utf-8") if isinstance(value, bytes) else value
    return value


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    return json_safe(obj)


def write_geojson(frame, path):
    features = []
    frame_wgs84 = frame.to_crs(4326)
    for _, row in frame_wgs84.iterrows():
        props = {}
        for col in frame_wgs84.columns:
            if col == "geometry":
                continue
            props[col] = json_safe(row[col])
        features.append(
            {
                "type": "Feature",
                "properties": props,
                "geometry": mapping(row.geometry),
            }
        )
    payload = {"type": "FeatureCollection", "name": path.stem, "features": features}
    path.write_text(json.dumps(payload, ensure_ascii=False, allow_nan=False), encoding="utf-8")


write_geojson(gdf, OUT_DATA / "blocks_clean.geojson")
gdf.drop(columns="geometry").to_csv(OUT_DATA / "indicators.csv", index=False, encoding="utf-8-sig")

print("Невалидных до очистки:", invalid_before)
print("Кварталов до очистки:", blocks_before_cleaning)
print("Кварталов после очистки:", blocks_after_cleaning)
print("Отброшено нулевых/линейных геометрий:", dropped_zero_area)
print("UTM CRS:", UTM_CRS)
print("Площадь, км²:", round(gdf["area_m2"].sum() / 1e6, 3))
print("Население, чел.:", int(gdf["population"].sum()))
print("Изменений морфотипа:", int(gdf["morphotype_changed"].sum()))
print("Сохранено:", OUT_DATA / "blocks_clean.geojson")
print("Сохранено:", OUT_DATA / "indicators.csv")


Невалидных до очистки: 6
Кварталов до очистки: 351
Кварталов после очистки: 349
Отброшено нулевых/линейных геометрий: 2
UTM CRS: EPSG:32636
Площадь, км²: 33.318
Население, чел.: 91521
Изменений морфотипа: 135
Сохранено: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\gatchina\blocks_clean.geojson
Сохранено: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\gatchina\indicators.csv


In [5]:
area_zero = int((gdf["area_m2"] <= 0).sum())
add_check(
    "Положительная площадь после очистки",
    "WARN" if area_zero else "OK",
    f"кварталов с нулевой/отрицательной площадью: {area_zero}",
)

add_check(
    "Геометрии после очистки",
    "WARN" if dropped_zero_area else "OK",
    f"осталось {blocks_after_cleaning} из {blocks_before_cleaning}; отброшено {dropped_zero_area} нулевых/линейных геометрий, которые нельзя показать на карте и использовать в плотностных расчётах",
)

add_check(
    "site_area / area_m2",
    "WARN",
    f"медиана {float(gdf['site_area_ratio'].median()):.4f}; исходная площадь в {float(gdf['site_area_ratio'].median()):.2f} раза больше метрической",
)
add_check(
    "site_area / area(3857)",
    "OK" if abs(float(gdf["site_area_3857_ratio"].median()) - 1.0) < 1e-6 else "WARN",
    f"медиана {float(gdf['site_area_3857_ratio'].median()):.6f}",
)

for name, actual, expected in [
    ("FSI: пересчёт = build_floor_area / area_m2", gdf["fsi_calc"], gdf["build_floor_area"] / gdf["area_m2"].replace(0, np.nan)),
    ("GSI: пересчёт = footprint_area / area_m2", gdf["gsi_calc"], gdf["footprint_area"] / gdf["area_m2"].replace(0, np.nan)),
    ("MXI после очистки", gdf["mxi"], gdf["living_area"] / gdf["build_floor_area"].replace(0, np.nan)),
    ("L после очистки", gdf["l"], gdf["build_floor_area"] / gdf["footprint_area"].replace(0, np.nan)),
]:
    err = median_rel_error(actual, expected)
    add_check(name, "OK" if (pd.notna(err) and err < 0.02) else "WARN", f"медианная относительная ошибка: {err:.4f}" if pd.notna(err) else "нет данных")

living_per_capita_median = float(gdf["living_per_capita"].median())
add_check(
    "Жилплощадь на человека",
    "OK" if 20 <= living_per_capita_median <= 45 else "WARN",
    f"медиана {living_per_capita_median:.1f} м²/чел. (ожидаемо около 30)",
)

unknown_land_after = int((gdf["land_use_ru"] == "Не определено").sum())
unknown_morph_after = int((gdf["morphotype_ru"] == "Не определено").sum())
add_check("Землепользование: не определено", "WARN" if unknown_land_after else "OK", f"кварталов без землепользования: {unknown_land_after}")
add_check("Морфотип: не определено", "WARN" if unknown_morph_after else "OK", f"кварталов без морфотипа: {unknown_morph_after}")

changed = int(gdf["morphotype_changed"].sum())
add_check(
    "Морфотипы: изменения после пересчёта",
    "WARN" if changed else "OK",
    f"изменилось у {changed} из {len(gdf)} кварталов; это чувствительность классификации к площади",
)

add_check(
    "Итоговая площадь",
    "OK",
    f"{gdf['area_m2'].sum() / 1e6:.2f} км² по метрической геометрии",
)
add_check(
    "Итоговое население",
    "OK",
    f"{int(gdf['population'].sum()):,} чел. (оценка)".replace(",", " "),
)

validation_df = pd.DataFrame(checks)
display(validation_df)
validation_df.to_csv(OUT_DATA / "validation_report.csv", index=False, encoding="utf-8-sig")
print("Отчёт проверок сохранён:", OUT_DATA / "validation_report.csv")


,Проверка,Статус,Комментарий
0,Обязательные колонки,OK,все обязательные колонки на месте
1,Число объектов,OK,351 кварталов
2,CRS исходного файла,OK,EPSG:4326
3,Типы геометрии,OK,Polygon: 351
4,Валидность геометрии,WARN,невалидных: 6 — будут исправлены через make_valid
5,Пустая геометрия,OK,пустых/None: 0
6,Дубли геометрии,OK,полных дублей: 0
7,Пропуски,WARN,"land_use: 7, share: 7, gsi: 2, morphotype: 55"
8,Диапазоны значений,OK,"отрицательных площадей и долей вне [0, 1] нет"
9,Категории землепользования,OK,6 ожидаемых категорий


Отчёт проверок сохранён: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\gatchina\validation_report.csv


## Сводные показатели и таблицы

Теперь собираем городские итоги и разрезы. Средние FSI/GSI считаются как среднее по кварталам, а не как отношение сумм: так один крупный квартал не перевешивает всю выборку. Площади и население суммируются.


In [6]:
def safe_mean(series):
    s = pd.to_numeric(pd.Series(series), errors="coerce").replace([np.inf, -np.inf], np.nan)
    return float(s.mean()) if s.notna().any() else np.nan


def ru_int(value):
    return f"{int(round(float(value))):,}".replace(",", " ")


def ru_num(value, digits=1):
    return f"{float(value):,.{digits}f}".replace(",", " ").replace(".", ",")


blocks_before_cleaning = int(globals().get("blocks_before_cleaning", len(gdf)))
blocks_dropped = blocks_before_cleaning - len(gdf)
total_area_km2 = float(gdf["area_m2"].sum() / 1e6)
source_area_km2 = float(gdf["site_area"].sum() / 1e6)
total_population = float(gdf["population"].sum())
total_floor_km2 = float(gdf["build_floor_area"].sum() / 1e6)
total_living_km2 = float(gdf["living_area"].sum() / 1e6)
avg_fsi_source = safe_mean(gdf["fsi"])
avg_gsi_source = safe_mean(gdf["gsi"])
avg_fsi_calc = safe_mean(gdf["fsi_calc"])
avg_gsi_calc = safe_mean(gdf["gsi_calc"])
avg_accessibility = safe_mean(gdf["area_accessibility"])
site_factor = float(gdf["site_area_ratio"].median())
changed_morph = int(gdf["morphotype_changed"].sum())

n_le10 = int((gdf["area_accessibility"] <= 10).sum())
n_gt15 = int((gdf["area_accessibility"] > 15).sum())
share_le10 = 100.0 * n_le10 / len(gdf)
share_gt15 = 100.0 * n_gt15 / len(gdf)

land_use_summary = (
    gdf.groupby("land_use_ru", dropna=False)
    .agg(
        blocks=("block_id", "size"),
        area_km2=("area_m2", lambda s: float(s.sum() / 1e6)),
        population=("population", "sum"),
        floor_km2=("build_floor_area", lambda s: float(s.sum() / 1e6)),
        avg_fsi=("fsi_calc", safe_mean),
        avg_gsi=("gsi_calc", safe_mean),
        avg_mxi=("mxi", safe_mean),
        avg_l=("l", safe_mean),
        avg_accessibility=("area_accessibility", safe_mean),
    )
    .reset_index()
    .sort_values("area_km2", ascending=False)
)

morph_summary = (
    gdf.groupby("morphotype_ru", dropna=False)
    .agg(
        blocks=("block_id", "size"),
        area_km2=("area_m2", lambda s: float(s.sum() / 1e6)),
        population=("population", "sum"),
        floor_km2=("build_floor_area", lambda s: float(s.sum() / 1e6)),
        avg_fsi=("fsi_calc", safe_mean),
        avg_gsi=("gsi_calc", safe_mean),
        avg_mxi=("mxi", safe_mean),
        avg_l=("l", safe_mean),
        avg_accessibility=("area_accessibility", safe_mean),
    )
    .reset_index()
    .sort_values("population", ascending=False)
)

top_pop = gdf.nlargest(10, "population")[
    ["block_id", "population", "area_ha", "land_use_ru", "morphotype_ru", "fsi_calc", "gsi_calc", "area_accessibility"]
].copy()
top_fsi = gdf.nlargest(10, "fsi_calc")[
    ["block_id", "fsi_calc", "gsi_calc", "l", "population", "area_ha", "land_use_ru", "morphotype_ru"]
].copy()

access_low = gdf.nsmallest(5, "area_accessibility")[
    ["block_id", "area_accessibility", "population", "land_use_ru", "morphotype_ru"]
].copy()
access_low["group"] = "Самые доступные"
access_high = gdf.nlargest(5, "area_accessibility")[
    ["block_id", "area_accessibility", "population", "land_use_ru", "morphotype_ru"]
].copy()
access_high["group"] = "Самые изолированные"
access_extremes = pd.concat([access_low, access_high], ignore_index=True)

morph_changes = (
    gdf.loc[gdf["morphotype_changed"], ["morphotype_ru", "morphotype_recalc"]]
    .value_counts()
    .reset_index(name="Кварталов")
    .sort_values("Кварталов", ascending=False)
)

access_order = ["до 10 мин", "10–15 мин", "15–20 мин", "более 20 мин", "Нет данных"]
access_summary = gdf["access_class"].value_counts().reindex(access_order, fill_value=0).astype(int).reset_index()
access_summary.columns = ["Класс доступности", "Кварталов"]
access_summary["Доля, %"] = (100.0 * access_summary["Кварталов"] / len(gdf)).round(1)

dominant_land_use = land_use_summary.iloc[0]
dominant_morph_pop = morph_summary.iloc[0]
top_block = gdf.loc[gdf["population"].idxmax()]

summary_df = pd.DataFrame(
    [
        ("Кварталов в исходном файле", blocks_before_cleaning, "шт."),
        ("Кварталов после очистки", len(gdf), "шт."),
        ("Отброшено нулевых геометрий", blocks_dropped, "шт."),
        ("Площадь территории (исправленная)", total_area_km2, "км²"),
        ("Площадь территории (как в файле)", source_area_km2, "км²"),
        ("Население (оценка)", total_population, "чел."),
        ("Общая площадь этажей", total_floor_km2, "км²"),
        ("Жилая площадь", total_living_km2, "км²"),
        ("Средний FSI (как в файле)", avg_fsi_source, "доля"),
        ("Средний FSI (пересчитанный)", avg_fsi_calc, "доля"),
        ("Средний GSI (как в файле)", avg_gsi_source, "доля"),
        ("Средний GSI (пересчитанный)", avg_gsi_calc, "доля"),
        ("Средняя доступность", avg_accessibility, "мин"),
        ("Кварталов с доступностью ≤ 10 мин", share_le10, "%"),
        ("Кварталов с доступностью > 15 мин", share_gt15, "%"),
        ("Коэффициент завышения site_area", site_factor, "раз"),
        ("Кварталов с изменённым морфотипом", changed_morph, "шт."),
    ],
    columns=["Показатель", "Значение", "Единица"],
)

conclusions = [
    f"В выборке {len(gdf)} кварталов общей площадью {ru_num(total_area_km2, 1)} км²; оценка населения — {ru_int(total_population)} чел.",
    f"По площади лидирует категория «{dominant_land_use['land_use_ru']}»: {ru_num(dominant_land_use['area_km2'], 1)} км², это {ru_num(100.0 * dominant_land_use['area_km2'] / total_area_km2, 1)}% территории.",
    f"Больше всего жителей — в морфотипе «{dominant_morph_pop['morphotype_ru']}»: {ru_int(dominant_morph_pop['population'])} чел.",
    f"Средняя доступность территории — {ru_num(avg_accessibility, 1)} мин; {n_le10} кварталов ({ru_num(share_le10, 1)}%) имеют доступность до 10 минут, {n_gt15} ({ru_num(share_gt15, 1)}%) — более 15 минут.",
    f"Самый населённый квартал — #{int(top_block['block_id'])}: {ru_int(top_block['population'])} чел., {ru_num(top_block['area_ha'], 1)} га, FSI {ru_num(top_block['fsi_calc'], 3)}, GSI {ru_num(top_block['gsi_calc'], 3)}.",
    f"Исходная site_area посчитана в EPSG:3857 и в {ru_num(site_factor, 2)} раза больше метрической площади. Поэтому исходные FSI/GSI занижены; в выводах используем пересчитанные значения, исходные показываем для сверки.",
    f"При пересчёте Strelka по исправленному FSI у {changed_morph} кварталов меняется категория. Это не ошибка данных, а чувствительность морфотипа к площади: классификация пересчитывает плотность на другой знаменатель.",
    f"После очистки осталось {len(gdf)} из {blocks_before_cleaning} кварталов: {blocks_dropped} геометрии схлопнулись в линии нулевой площади и не могут участвовать в плотностных расчётах.",
    f"Ограничения: население — оценка по жилплощади и нормативу, а не официальная статистика; без землепользования — {unknown_land_after} кварталов, без морфотипа — {unknown_morph_after}; школы, обеспеченность и доступность матричным методом не считались, потому что этих полей нет в исходном файле.",
]

summary_json = {
    "city": CITY,
    "region": REGION,
    "source": str(SOURCE),
    "generated": date.today().isoformat(),
    "blocks": len(gdf),
    "blocks_source": blocks_before_cleaning,
    "blocks_dropped": blocks_dropped,
    "area_km2": total_area_km2,
    "source_area_km2": source_area_km2,
    "population": total_population,
    "floor_km2": total_floor_km2,
    "living_km2": total_living_km2,
    "avg_fsi_source": avg_fsi_source,
    "avg_fsi_calc": avg_fsi_calc,
    "avg_gsi_source": avg_gsi_source,
    "avg_gsi_calc": avg_gsi_calc,
    "avg_accessibility": avg_accessibility,
    "share_access_le10": share_le10,
    "share_access_gt15": share_gt15,
    "site_area_factor": site_factor,
    "morphotype_changes": changed_morph,
    "conclusions": conclusions,
}

land_use_summary.to_csv(OUT_DATA / "land_use_summary.csv", index=False, encoding="utf-8-sig")
morph_summary.to_csv(OUT_DATA / "morphotype_summary.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv(OUT_DATA / "summary.csv", index=False, encoding="utf-8-sig")
(OUT_DATA / "summary.json").write_text(json.dumps(to_jsonable(summary_json), ensure_ascii=False, indent=2), encoding="utf-8")

display(summary_df)
display(land_use_summary)
display(morph_summary)
print("Сводки сохранены в", OUT_DATA)


,Показатель,Значение,Единица
0,Кварталов в исходном файле,351.000000,шт.
1,Кварталов после очистки,349.000000,шт.
2,Отброшено нулевых геометрий,2.000000,шт.
3,Площадь территории (исправленная),33.317809,км²
4,Площадь территории (как в файле),129.465326,км²
5,Население (оценка),91521.000000,чел.
6,Общая площадь этажей,6.236293,км²
7,Жилая площадь,2.907912,км²
8,Средний FSI (как в файле),0.053187,доля
9,Средний FSI (пересчитанный),0.206633,доля


,land_use_ru,blocks,area_km2,population,floor_km2,avg_fsi,avg_gsi,avg_mxi,avg_l,avg_accessibility
1,Жилая,211,11.678572,83320.0,3.899384,0.263051,0.121543,0.669736,1.886959,9.686849
4,Рекреационная,34,8.728344,924.0,0.110963,0.016234,0.013275,0.039401,0.430086,12.483686
3,Промышленная,19,4.815994,449.0,1.193300,0.190941,0.099970,0.178137,1.428396,11.538070
0,Деловая,50,3.398600,6514.0,0.905073,0.233101,0.126459,0.309179,1.408460,9.115079
5,Специальная,9,3.020522,32.0,0.035561,0.005914,0.003342,0.025143,0.871572,10.886166
6,Транспортная,21,1.662476,265.0,0.091364,0.029574,0.023803,0.127991,0.808512,9.574328
2,Не определено,5,0.013301,17.0,0.000649,0.020425,0.020425,0.320000,0.400000,14.215537


,morphotype_ru,blocks,area_km2,population,floor_km2,avg_fsi,avg_gsi,avg_mxi,avg_l,avg_accessibility
4,Среднеэтажная,23,2.810271,54702.0,2.442443,0.867345,0.158143,0.718338,5.573837,9.452163
1,Малоэтажная модель,25,1.955551,19749.0,1.179384,0.633277,0.213573,0.528928,3.038768,8.586062
0,Индивидуальная жилая застройка,199,10.849316,16047.0,0.924632,0.137069,0.105499,0.681872,1.279656,9.891199
5,Среднеэтажный микрорайон,1,0.168309,660.0,0.052862,0.314079,0.076221,0.396726,4.120634,9.918830
2,Малоэтажная нежилая,48,15.058919,363.0,1.636971,0.182156,0.102170,0.003695,1.546186,10.253439
3,Не определено,53,2.475442,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,11.527886


Сводки сохранены в C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\gatchina


## Графики

Графики отвечают на разные вопросы:

- **землепользование** — где находятся жилые, деловые, рекреационные и промышленные территории;
- **доступность** — насколько кварталы близки к точкам притяжения;
- **FSI/GSI** — насколько интенсивно застроена территория и как сильно исходные значения отличаются от пересчитанных;
- **морфотипы** — какие типы застройки преобладают по количеству кварталов и по населению;
- **FSI × этажность** — есть ли связь между интенсивностью и высотностью;
- **топ-10** — какие кварталы дают основной вклад в население.


In [7]:
MORPH_ORDER_RU = [
    "Индивидуальная жилая застройка",
    "Малоэтажная модель",
    "Малоэтажная нежилая",
    "Среднеэтажная",
    "Среднеэтажный микрорайон",
    "Не определено",
]
MORPH_COLORS = {
    "Индивидуальная жилая застройка": "#4e79a7",
    "Малоэтажная модель": "#f28e2b",
    "Малоэтажная нежилая": "#e15759",
    "Среднеэтажная": "#76b7b2",
    "Среднеэтажный микрорайон": "#59a14f",
    "Не определено": "#bab0ac",
}
LAND_USE_ORDER_RU = [
    "Жилая", "Деловая", "Рекреационная", "Промышленная",
    "Транспортная", "Специальная", "Сельскохозяйственная",
    "Не определено", "Другое",
]
LAND_USE_COLORS = {
    "Жилая": "#4e79a7",
    "Деловая": "#f28e2b",
    "Рекреационная": "#59a14f",
    "Промышленная": "#e15759",
    "Транспортная": "#76b7b2",
    "Специальная": "#b07aa1",
    "Сельскохозяйственная": "#edc948",
    "Не определено": "#bab0ac",
    "Другое": "#9c755f",
}


def hist_data(values, bins=18, vmin=None, vmax=None):
    s = pd.to_numeric(pd.Series(values), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return {"labels": [], "counts": []}
    lo = float(s.min()) if vmin is None else float(vmin)
    hi = float(s.max()) if vmax is None else float(vmax)
    if hi <= lo:
        hi = lo + 1.0
    counts, edges = np.histogram(s, bins=bins, range=(lo, hi))
    labels = [f"{edges[i]:.2f}–{edges[i + 1]:.2f}" for i in range(len(counts))]
    return {"labels": labels, "counts": counts.astype(int).tolist()}


landuse_chart = land_use_summary.sort_values("area_km2", ascending=False).reset_index(drop=True)
morph_counts = morph_summary.set_index("morphotype_ru")["blocks"].reindex(MORPH_ORDER_RU, fill_value=0)
morph_population = morph_summary.set_index("morphotype_ru")["population"].reindex(MORPH_ORDER_RU, fill_value=0)
source_morph_counts = gdf["morphotype_ru"].value_counts().reindex(MORPH_ORDER_RU, fill_value=0)
recalc_morph_counts = gdf["morphotype_recalc"].value_counts().reindex(MORPH_ORDER_RU, fill_value=0)

scatter_datasets = []
for morph_name in MORPH_ORDER_RU:
    sub = gdf[(gdf["morphotype_ru"] == morph_name) & gdf["fsi_calc"].notna() & gdf["l"].notna()]
    if sub.empty:
        continue
    points = []
    for _, row in sub.iterrows():
        points.append(
            {
                "x": round(float(row["fsi_calc"]), 4),
                "y": round(float(row["l"]), 3),
                "id": int(row["block_id"]),
                "population": int(row["population"]) if pd.notna(row["population"]) else 0,
            }
        )
    scatter_datasets.append(
        {
            "label": morph_name,
            "data": points,
            "backgroundColor": MORPH_COLORS.get(morph_name, "#999999"),
        }
    )

top_pop_chart = top_pop.sort_values("population", ascending=True)
chart_data = {
    "landuse": {
        "labels": landuse_chart["land_use_ru"].tolist(),
        "area_ha": [round(float(x), 1) for x in landuse_chart["area_km2"] * 100.0],
        "population": [int(x) for x in landuse_chart["population"].fillna(0)],
        "colors": [LAND_USE_COLORS.get(x, "#999999") for x in landuse_chart["land_use_ru"]],
    },
    "access": {
        "labels": access_summary["Класс доступности"].tolist(),
        "counts": [int(x) for x in access_summary["Кварталов"]],
    },
    "fsi_compare": {
        "labels": ["FSI", "GSI"],
        "source": [round(avg_fsi_source, 4), round(avg_gsi_source, 4)],
        "calc": [round(avg_fsi_calc, 4), round(avg_gsi_calc, 4)],
    },
    "fsi_hist": hist_data(gdf["fsi_calc"], bins=18, vmin=0.0, vmax=max(0.1, float(gdf["fsi_calc"].max()))),
    "gsi_hist": hist_data(gdf["gsi_calc"], bins=15, vmin=0.0, vmax=max(0.1, float(gdf["gsi_calc"].max()))),
    "scatter": {"datasets": scatter_datasets},
    "morph": {
        "labels": MORPH_ORDER_RU,
        "counts": [int(x) for x in morph_counts],
        "population": [int(x) for x in morph_population.fillna(0)],
        "source_counts": [int(x) for x in source_morph_counts],
        "recalc_counts": [int(x) for x in recalc_morph_counts],
        "colors": [MORPH_COLORS[x] for x in MORPH_ORDER_RU],
    },
    "top_pop": {
        "labels": [f"#{int(x)}" for x in top_pop_chart["block_id"]],
        "values": [int(x) for x in top_pop_chart["population"]],
    },
    "colors": {
        "morph": MORPH_COLORS,
        "landuse": LAND_USE_COLORS,
    },
}

(OUT_DATA / "chart_data.json").write_text(
    json.dumps(to_jsonable(chart_data), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Графики подготовлены:", ", ".join(chart_data.keys()))
print("Точек на диаграмме FSI × L:", sum(len(d["data"]) for d in scatter_datasets))
print("Сохранено:", OUT_DATA / "chart_data.json")


Графики подготовлены: landuse, access, fsi_compare, fsi_hist, gsi_hist, scatter, morph, top_pop, colors
Точек на диаграмме FSI × L: 349
Сохранено: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\data\gatchina\chart_data.json


## Сборка одной HTML-страницы

Страница состоит из шести блоков: краткие выводы, интерактивная карта, графики, таблицы, методика и проверки, ограничения. Все данные и библиотеки встроены в один файл. Подложка OSM — единственный внешний ресурс; без интернета карта останется с полигонами кварталов.


In [8]:
def fmt_int(value):
    if value is None or pd.isna(value):
        return "—"
    return f"{int(round(float(value))):,}".replace(",", " ")


def fmt_num(value, digits=2):
    if value is None or pd.isna(value):
        return "—"
    return f"{float(value):,.{digits}f}".replace(",", " ").replace(".", ",")


def fmt_frac(value):
    return fmt_num(value, 3)


def fmt_km2(value):
    return fmt_num(value, 2)


def fmt_ha(value):
    return fmt_num(value, 1)


def fmt_pct(value):
    return f"{fmt_num(value, 1)} %"


def fmt_any(value):
    if value is None or pd.isna(value):
        return "—"
    if isinstance(value, (int, np.integer)) and not isinstance(value, bool):
        return fmt_int(value)
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return fmt_int(value)
    return fmt_num(value, 3)


def badge(status):
    cls = {"OK": "ok", "WARN": "warn", "ERROR": "error"}.get(status, "")
    label = {"OK": "OK", "WARN": "Внимание", "ERROR": "Ошибка"}.get(status, status)
    return f'<span class="badge {cls}">{html.escape(str(label))}</span>'


def table_html(frame, columns, headers, formatters=None, empty_text="Нет данных"):
    formatters = formatters or {}
    if frame is None or len(frame) == 0:
        return f'<div class="empty">{html.escape(empty_text)}</div>'
    rows = []
    for _, row in frame.iterrows():
        cells = []
        for col in columns:
            value = row[col]
            if col in formatters:
                text = formatters[col](value)
            elif isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
                text = fmt_num(value, 2)
            elif value is None or (not isinstance(value, (list, tuple, dict)) and pd.isna(value)):
                text = "—"
            else:
                text = html.escape(str(value))
            align = "num" if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool) else ""
            cells.append(f'<td class="{align}">{text}</td>')
        rows.append("<tr>" + "".join(cells) + "</tr>")
    head = "".join(f"<th>{html.escape(str(h))}</th>" for h in headers)
    return f'<div class="table-wrap"><table><thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>'


def quality_table(frame):
    rows = []
    for _, row in frame.iterrows():
        rows.append(
            "<tr>"
            f"<td>{html.escape(str(row['Проверка']))}</td>"
            f"<td>{badge(str(row['Статус']))}</td>"
            f"<td>{html.escape(str(row['Комментарий']))}</td>"
            "</tr>"
        )
    return (
        '<div class="table-wrap"><table><thead><tr><th>Проверка</th><th>Статус</th><th>Комментарий</th></tr></thead>'
        f'<tbody>{"".join(rows)}</tbody></table></div>'
    )


kpi_items = [
    ("Кварталов", fmt_int(len(gdf)), "в выборке"),
    ("Площадь", f"{total_area_km2:.1f} км²", "метрическая"),
    ("Население", fmt_int(total_population), "оценка"),
    ("Средний FSI", fmt_frac(avg_fsi_calc), "пересчитанный"),
    ("Средний GSI", fmt_frac(avg_gsi_calc), "пересчитанный"),
    ("Доступность", f"{avg_accessibility:.1f} мин", "в среднем"),
]
kpi_cards = "".join(
    f'<div class="kpi"><span>{html.escape(label)}</span><b>{html.escape(value)}</b><small>{html.escape(note)}</small></div>'
    for label, value, note in kpi_items
)
conclusions_html = "".join(f"<li>{html.escape(item)}</li>" for item in conclusions)

metric_rows = [
    ("Землепользование", "категория", "Преобладающая функция квартала по OSM", "Где жилая, деловая, рекреационная и промышленная территория"),
    ("Доли землепользования", "0–1", "Доли каждого типа использования в площади квартала", "Насколько квартал смешанный"),
    ("site_area", "м²", "Площадь квартала в исходном файле", "Для сверки; в этом файле завышена"),
    ("area_m2", "м²", "Площадь в метрической проекции UTM 36N", "Честная физическая площадь"),
    ("footprint_area", "м²", "Площадь застройки в плане", "Сколько земли занято зданиями"),
    ("build_floor_area", "м²", "Суммарная поэтажная площадь", "Объём застройки"),
    ("living_area", "м²", "Жилая площадь", "Жилищный фонд"),
    ("non_living_area", "м²", "Нежилая площадь", "Деловая и сервисная нагрузка"),
    ("population", "чел.", "Оценка населения", "Где живут люди"),
    ("population_density", "чел./га", "Население на гектар", "Плотность заселения"),
    ("fsi / fsi_calc", "доля", "Поэтажная площадь / площадь квартала", "Интенсивность застройки; `fsi_calc` — на честную площадь"),
    ("gsi / gsi_calc", "доля", "Площадь застройки / площадь квартала", "Доля застроенной территории; `gsi_calc` — на честную площадь"),
    ("mxi", "доля", "Жилая площадь / поэтажная площадь", "Насколько квартал жилой"),
    ("l", "этажей", "Поэтажная площадь / площадь застройки", "Средняя высотность"),
    ("area_accessibility", "мин", "Оценка доступности территории", "Удалённость от точек притяжения"),
    ("morphotype", "категория", "Морфотип Strelka по плотностным индикаторам", "Тип застройки"),
    ("living_per_capita", "м²/чел.", "Жилая площадь на человека", "Жилищная обеспеченность"),
    ("floor_per_capita", "м²/чел.", "Поэтажная площадь на человека", "Общая обеспеченность площадью"),
]
metric_table = (
    '<div class="table-wrap"><table><thead><tr><th>Показатель</th><th>Единица</th><th>Что означает</th><th>Что помогает узнать</th></tr></thead><tbody>'
    + "".join(
        f"<tr><td><code>{html.escape(a)}</code></td><td>{html.escape(b)}</td><td>{html.escape(c)}</td><td>{html.escape(d)}</td></tr>"
        for a, b, c, d in metric_rows
    )
    + "</tbody></table></div>"
)

summary_table = table_html(
    summary_df,
    ["Показатель", "Значение", "Единица"],
    ["Показатель", "Значение", "Единица"],
    {"Значение": fmt_any},
)

land_use_table = table_html(
    land_use_summary,
    ["land_use_ru", "blocks", "area_km2", "population", "floor_km2", "avg_fsi", "avg_gsi", "avg_mxi", "avg_l", "avg_accessibility"],
    ["Землепользование", "Кварталов", "Площадь, км²", "Население, чел.", "Этажи, км²", "FSI", "GSI", "MXI", "L", "Доступность, мин"],
    {"area_km2": fmt_km2, "population": fmt_int, "floor_km2": fmt_km2, "avg_fsi": fmt_frac, "avg_gsi": fmt_frac, "avg_mxi": fmt_frac, "avg_l": lambda v: fmt_num(v, 2), "avg_accessibility": lambda v: fmt_num(v, 1)},
)

morph_table = table_html(
    morph_summary,
    ["morphotype_ru", "blocks", "area_km2", "population", "floor_km2", "avg_fsi", "avg_gsi", "avg_mxi", "avg_l", "avg_accessibility"],
    ["Морфотип", "Кварталов", "Площадь, км²", "Население, чел.", "Этажи, км²", "FSI", "GSI", "MXI", "L", "Доступность, мин"],
    {"area_km2": fmt_km2, "population": fmt_int, "floor_km2": fmt_km2, "avg_fsi": fmt_frac, "avg_gsi": fmt_frac, "avg_mxi": fmt_frac, "avg_l": lambda v: fmt_num(v, 2), "avg_accessibility": lambda v: fmt_num(v, 1)},
)

top_pop_table = table_html(
    top_pop,
    ["block_id", "population", "area_ha", "land_use_ru", "morphotype_ru", "fsi_calc", "gsi_calc", "area_accessibility"],
    ["Квартал", "Население, чел.", "Площадь, га", "Землепользование", "Морфотип", "FSI", "GSI", "Доступность, мин"],
    {"block_id": lambda v: f"#{int(v)}", "population": fmt_int, "area_ha": fmt_ha, "fsi_calc": fmt_frac, "gsi_calc": fmt_frac, "area_accessibility": lambda v: fmt_num(v, 1)},
)

top_fsi_table = table_html(
    top_fsi,
    ["block_id", "fsi_calc", "gsi_calc", "l", "population", "area_ha", "land_use_ru", "morphotype_ru"],
    ["Квартал", "FSI", "GSI", "L", "Население, чел.", "Площадь, га", "Землепользование", "Морфотип"],
    {"block_id": lambda v: f"#{int(v)}", "fsi_calc": fmt_frac, "gsi_calc": fmt_frac, "l": lambda v: fmt_num(v, 2), "population": fmt_int, "area_ha": fmt_ha},
)

access_table = table_html(
    access_extremes,
    ["group", "block_id", "area_accessibility", "population", "land_use_ru", "morphotype_ru"],
    ["Группа", "Квартал", "Доступность, мин", "Население, чел.", "Землепользование", "Морфотип"],
    {"block_id": lambda v: f"#{int(v)}", "area_accessibility": lambda v: fmt_num(v, 1), "population": fmt_int},
)

morph_changes_table = table_html(
    morph_changes,
    ["morphotype_ru", "morphotype_recalc", "Кварталов"],
    ["Морфотип в файле", "Морфотип после пересчёта", "Кварталов"],
    {"Кварталов": fmt_int},
    empty_text="Категории не менялись.",
)

print("HTML-компоненты подготовлены.")
print("KPI-карточек:", len(kpi_items))
print("Строк в таблице землепользования:", len(land_use_summary))
print("Строк в таблице морфотипов:", len(morph_summary))


HTML-компоненты подготовлены.
KPI-карточек: 6
Строк в таблице землепользования: 7
Строк в таблице морфотипов: 6


In [9]:
HTML_TEMPLATE_TOP = r"""<!doctype html>
<html lang="ru">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__CITY__ — индикаторы кварталов</title>
<style>__LEAFLET_CSS__</style>
<style>
:root{
  --bg:#f4f6f8; --card:#ffffff; --text:#17212b; --muted:#5b6b7c; --line:#dce3ea;
  --accent:#2f6f8f; --accent2:#d99b3d; --good:#2e7d32; --warn:#a86400; --bad:#b3261e;
  --shadow:0 10px 30px rgba(23,33,43,.08);
}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif;background:var(--bg);color:var(--text);line-height:1.55}
a{color:var(--accent)}
.wrap{max-width:1240px;margin:0 auto;padding:0 20px}
.hero{background:linear-gradient(135deg,#15384b 0%,#2f6f8f 58%,#d99b3d 140%);color:#fff;padding:44px 0 30px}
.eyebrow{font-size:12px;letter-spacing:.14em;text-transform:uppercase;opacity:.8;font-weight:700}
h1{font-size:clamp(30px,5vw,54px);line-height:1.06;margin:10px 0 12px;letter-spacing:-.03em}
.hero p{max-width:820px;font-size:17px;opacity:.92;margin:0 0 8px}
.hero .source{font-size:13px;opacity:.75}
.kpi-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-top:26px}
.kpi{background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.22);border-radius:14px;padding:14px 16px;backdrop-filter:blur(5px)}
.kpi span{display:block;font-size:12px;text-transform:uppercase;letter-spacing:.06em;opacity:.8}
.kpi b{display:block;font-size:clamp(22px,2.4vw,32px);line-height:1.15;margin:4px 0 2px}
.kpi small{opacity:.75}
.toc{position:sticky;top:0;z-index:1000;background:rgba(255,255,255,.96);border-bottom:1px solid var(--line);backdrop-filter:blur(8px)}
.toc .wrap{display:flex;gap:4px;overflow-x:auto;padding-top:8px;padding-bottom:8px}
.toc a{white-space:nowrap;text-decoration:none;color:var(--muted);font-size:14px;font-weight:650;padding:8px 11px;border-radius:9px}
.toc a:hover{background:#eef4f7;color:var(--accent)}
.section{padding:34px 0 10px}
.section h2{font-size:clamp(24px,3.2vw,34px);line-height:1.15;margin:0 0 8px;letter-spacing:-.02em}
.section .lead{color:var(--muted);max-width:920px;margin:0 0 18px}
.card{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:18px 20px;box-shadow:var(--shadow)}
.card h3{margin:0 0 10px;font-size:18px}
.grid-2{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:18px}
.grid-3{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:18px}
ul.clean{margin:0;padding-left:20px}
ul.clean li{margin:0 0 9px}
.callout{border-left:4px solid var(--accent2);background:#fff8e9;border-radius:12px;padding:14px 16px;margin:18px 0;color:#4f3b16}
.callout b{color:#3d2c0e}
code{background:#eef2f5;border:1px solid #e1e7ed;border-radius:5px;padding:1px 5px;font-size:.92em}
.controls{display:flex;flex-wrap:wrap;gap:10px;align-items:end;margin:0 0 14px}
.control{display:flex;flex-direction:column;gap:4px;min-width:190px}
.control label{font-size:12px;color:var(--muted);font-weight:700;text-transform:uppercase;letter-spacing:.05em}
select,button,input[type="range"]{font:inherit}
select,button{background:#fff;border:1px solid var(--line);border-radius:10px;padding:9px 11px;color:var(--text)}
button{cursor:pointer;font-weight:650}
button:hover{border-color:var(--accent);color:var(--accent)}
.check{display:flex;align-items:center;gap:7px;padding:9px 0;font-size:14px;color:var(--muted)}
.map-shell{position:relative;height:640px;border-radius:16px;overflow:hidden;border:1px solid var(--line);box-shadow:var(--shadow);background:#dfe7ec}
#map{height:100%;width:100%}
.map-count{position:absolute;left:14px;top:14px;z-index:600;background:rgba(255,255,255,.96);border:1px solid var(--line);border-radius:10px;padding:7px 10px;font-size:13px;font-weight:650;box-shadow:0 4px 14px rgba(0,0,0,.08)}
.legend{position:absolute;right:14px;bottom:14px;z-index:600;background:rgba(255,255,255,.97);border:1px solid var(--line);border-radius:12px;padding:11px 13px;max-width:280px;font-size:12px;box-shadow:0 6px 20px rgba(0,0,0,.12)}
.legend h4{margin:0 0 8px;font-size:12px;text-transform:uppercase;letter-spacing:.05em;color:var(--muted)}
.legend-item{display:flex;align-items:center;gap:7px;margin:4px 0}
.swatch{width:15px;height:15px;border-radius:4px;border:1px solid rgba(0,0,0,.12);flex:0 0 auto}
.legend-note{margin-top:8px;color:var(--muted);font-size:11px;line-height:1.35}
.chart-card{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:16px 18px;box-shadow:var(--shadow);min-width:0}
.chart-head h3{margin:0 0 3px;font-size:17px}
.chart-head p{margin:0 0 12px;color:var(--muted);font-size:13px}
.chart-box{position:relative;height:300px;min-width:0}
.chart-box.tall{height:360px}
.table-section{margin:18px 0 26px}
.table-section h3{font-size:19px;margin:0 0 8px}
.table-section p{color:var(--muted);margin:0 0 10px;font-size:14px}
.table-wrap{overflow:auto;border:1px solid var(--line);border-radius:12px;background:#fff}
table{border-collapse:collapse;width:100%;font-size:13px}
th,td{padding:9px 10px;border-bottom:1px solid #edf1f5;text-align:left;vertical-align:top}
th{position:sticky;top:0;background:#f0f5f8;color:#415261;font-size:11px;text-transform:uppercase;letter-spacing:.04em;z-index:1}
tbody tr:nth-child(even){background:#fafbfc}
tbody tr:hover{background:#f4f9fb}
td.num{text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums}
.badge{display:inline-block;border-radius:999px;padding:2px 8px;font-size:11px;font-weight:800;letter-spacing:.03em}
.badge.ok{background:#e7f5e9;color:#1b5e20}
.badge.warn{background:#fff3d6;color:#7a4b00}
.badge.error{background:#fde8e7;color:#8e1b16}
.empty{padding:18px;color:var(--muted);background:#fff;border:1px dashed var(--line);border-radius:12px}
details{background:#fff;border:1px solid var(--line);border-radius:12px;padding:12px 14px;margin:10px 0}
summary{cursor:pointer;font-weight:700;color:#314657}
details p,details ul{color:var(--muted)}
.footer{margin-top:40px;padding:26px 0 40px;border-top:1px solid var(--line);color:var(--muted);font-size:13px}
.footer b{color:var(--text)}
.leaflet-tooltip{font-size:12px;line-height:1.45}
.tt-label{color:#5b6b7c}
.popup-table{border-collapse:collapse;width:100%;font-size:12px}
.popup-table td{padding:3px 4px;border-bottom:1px solid #edf1f5}
.popup-table td:first-child{color:#5b6b7c;padding-right:8px}
@media (max-width:900px){.grid-2,.grid-3{grid-template-columns:1fr}.map-shell{height:520px}}
@media print{.toc{position:static}.hero{background:#15384b}.controls{display:none}.map-shell{height:420px}}
</style>
</head>
<body>
<header class="hero">
  <div class="wrap">
    <div class="eyebrow">BlocksNet · агрегированные индикаторы кварталов</div>
    <h1>__CITY__: город в цифрах</h1>
    <p>Одна страница о кварталах Гатчины: население, плотность и этажность застройки, землепользование, доступность и морфотипы. Все расчёты и проверки выполнены в ноутбуке.</p>
    <p class="source">__SOURCE_NOTE__ · обновлено __UPDATED__ · город: __CITY__, __REGION__</p>
    <div class="kpi-grid">__KPI_CARDS__</div>
  </div>
</header>

<nav class="toc">
  <div class="wrap">
    <a href="#summary">Коротко</a>
    <a href="#map">Карта</a>
    <a href="#charts">Графики</a>
    <a href="#tables">Таблицы</a>
    <a href="#method">Методика</a>
  </div>
</nav>

<main class="wrap">
  <section id="summary" class="section">
    <h2>Коротко о городе</h2>
    <p class="lead">Сначала — главные выводы, затем — карта и детали. Все числа ниже относятся к __N_BLOCKS__ кварталам после очистки (в исходном файле было __N_SOURCE__), а не ко всей территории муниципального образования.</p>
    <div class="grid-2">
      <div class="card">
        <h3>Главные выводы</h3>
        <ul class="clean">__CONCLUSIONS__</ul>
      </div>
      <div class="card">
        <h3>Как читать страницу</h3>
        <ul class="clean">
          <li><b>Карта</b> показывает выбранный показатель по кварталам. Можно переключить метрику, отфильтровать землепользование и морфотип, навести курсор на квартал.</li>
          <li><b>Графики</b> дают распределения и сравнения. Наведите курсор, чтобы увидеть точные значения.</li>
          <li><b>Таблицы</b> содержат сводки по землепользованию и морфотипам, топ-10 и проверки качества.</li>
          <li><b>Методика</b> объясняет единицы, источник и ограничения.</li>
        </ul>
      </div>
    </div>
    <div class="callout">__CALLOUT__</div>
  </section>

  <section id="map" class="section">
    <h2>Интерактивная карта</h2>
    <p class="lead">Выберите показатель, при необходимости отфильтруйте кварталы. Цвет показывает значение: для числовых метрик — пять квантильных классов, для категорий — отдельный цвет. Серый цвет означает «нет данных».</p>
    <div class="controls">
      <div class="control">
        <label for="metric-select">Показатель на карте</label>
        <select id="metric-select"></select>
      </div>
      <div class="control">
        <label for="filter-landuse">Землепользование</label>
        <select id="filter-landuse"></select>
      </div>
      <div class="control">
        <label for="filter-morph">Морфотип</label>
        <select id="filter-morph"></select>
      </div>
      <div class="control" style="min-width:150px">
        <label for="opacity">Прозрачность</label>
        <input id="opacity" type="range" min="25" max="100" value="82">
      </div>
      <label class="check"><input id="basemap" type="checkbox" checked> Подложка OSM</label>
      <button id="reset-map" type="button">Сбросить карту</button>
    </div>
    <div class="map-shell">
      <div id="map"></div>
      <div id="map-count" class="map-count">Показано кварталов: —</div>
      <div id="map-legend" class="legend"></div>
    </div>
  </section>

  <section id="charts" class="section">
    <h2>Графики</h2>
    <p class="lead">Графики построены по очищенным данным. Для FSI/GSI используем пересчитанные значения; исходные показаны отдельно, чтобы была видна поправка на площадь.</p>
    <div class="grid-2">
      <div class="chart-card">
        <div class="chart-head"><h3>Площадь по землепользованию</h3><p>Гектары; сортировка по площади.</p></div>
        <div class="chart-box"><canvas id="chart-landuse-area"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Население по землепользованию</h3><p>Человек; где живёт основная часть жителей.</p></div>
        <div class="chart-box"><canvas id="chart-landuse-pop"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Доступность территории</h3><p>Распределение кварталов по классам доступности, минуты.</p></div>
        <div class="chart-box"><canvas id="chart-access"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>FSI и GSI: файл против пересчёта</h3><p>Средние по кварталам; пересчёт на метрическую площадь.</p></div>
        <div class="chart-box"><canvas id="chart-fsi-compare"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Распределение FSI</h3><p>Пересчитанный FSI по кварталам.</p></div>
        <div class="chart-box"><canvas id="chart-fsi-hist"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Распределение GSI</h3><p>Пересчитанный GSI по кварталам.</p></div>
        <div class="chart-box"><canvas id="chart-gsi-hist"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>FSI и средняя этажность</h3><p>Каждая точка — квартал; цвет — морфотип.</p></div>
        <div class="chart-box tall"><canvas id="chart-scatter"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Морфотипы: количество кварталов</h3><p>Сколько кварталов в каждой категории.</p></div>
        <div class="chart-box"><canvas id="chart-morph-count"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Морфотипы: население</h3><p>Где живут люди по типам застройки.</p></div>
        <div class="chart-box"><canvas id="chart-morph-pop"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Морфотипы: файл и пересчёт</h3><p>Сравнение исходной и пересчитанной классификации.</p></div>
        <div class="chart-box"><canvas id="chart-morph-compare"></canvas></div>
      </div>
      <div class="chart-card">
        <div class="chart-head"><h3>Топ-10 кварталов по населению</h3><p>Человек; подписи — номера кварталов.</p></div>
        <div class="chart-box"><canvas id="chart-top-pop"></canvas></div>
      </div>
    </div>
  </section>
"""


In [10]:
HTML_TEMPLATE_BOTTOM = r"""
  <section id="tables" class="section">
    <h2>Таблицы</h2>
    <p class="lead">Сводки по землепользованию и морфотипам, топ-10 кварталов и проверки качества. Кнопка ниже скачивает CSV с показателями всех кварталов.</p>
    <div class="table-section">
      <h3>Городские итоги</h3>
      <p>Ключевые суммы и средние по очищенной выборке.</p>
      __SUMMARY_TABLE__
    </div>
    <div class="table-section">
      <h3>Сводка по землепользованию</h3>
      <p>Площадь, население и средние плотностные показатели по категориям.</p>
      __LAND_USE_TABLE__
    </div>
    <div class="table-section">
      <h3>Сводка по морфотипам</h3>
      <p>Морфотип из исходного файла; для сравнения с пересчётом смотрите отдельную таблицу ниже.</p>
      __MORPH_TABLE__
    </div>
    <div class="table-section">
      <h3>Топ-10 по населению</h3>
      <p>Кварталы с наибольшей оценкой населения.</p>
      __TOP_POP_TABLE__
    </div>
    <div class="table-section">
      <h3>Топ-10 по FSI</h3>
      <p>Наиболее интенсивно застроенные кварталы по пересчитанному FSI.</p>
      __TOP_FSI_TABLE__
    </div>
    <div class="table-section">
      <h3>Доступность: крайние случаи</h3>
      <p>Пять самых доступных и пять самых изолированных кварталов.</p>
      __ACCESS_TABLE__
    </div>
    <div class="table-section">
      <h3>Изменения морфотипов</h3>
      <p>Категории, которые изменились после пересчёта Strelka на исправленную площадь.</p>
      __MORPH_CHANGES_TABLE__
    </div>
    <p><button type="button" onclick="downloadCSV()">Скачать CSV по кварталам</button></p>
  </section>

  <section id="method" class="section">
    <h2>Методика и проверки</h2>
    <p class="lead">Что именно посчитано, какие данные проверены и где границы интерпретации.</p>

    <div class="card" style="margin-bottom:18px">
      <h3>Показатели</h3>
      __METRIC_TABLE__
    </div>

    <div class="card" style="margin-bottom:18px">
      <h3>Проверки качества данных</h3>
      <p style="color:var(--muted);margin:0 0 10px">Проверки выполняются в ноутбуке. «Внимание» — это известное ограничение или особенность файла, а не остановка расчёта.</p>
      __QUALITY_TABLE__
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>Соответствие pipeline BlocksNet</h3>
        <div class="table-wrap">
          <table>
            <thead><tr><th>Шаг pipeline</th><th>В этом ноутбуке</th></tr></thead>
            <tbody>
              <tr><td>Скачивание OSM и нарезка кварталов</td><td>Не повторяется: готовый GeoJSON уже содержит кварталы</td></tr>
              <tr><td>Землепользование</td><td>Готовое поле + нормализация категорий</td></tr>
              <tr><td>Агрегирование зданий</td><td>Готовые площади, население, FSI/GSI/MXI/L</td></tr>
              <tr><td>Плотностные индикаторы</td><td>Исходные + пересчитанные на метрическую площадь</td></tr>
              <tr><td>Морфотипы Strelka</td><td>Исходные + пересчёт по исправленному FSI/GSI</td></tr>
              <tr><td>Доступность территории</td><td>Готовое поле <code>area_accessibility</code></td></tr>
              <tr><td>Матрица доступности</td><td>Не строится: нет исходных точек и матрицы</td></tr>
              <tr><td>Школы и обеспеченность</td><td>Не считаются: нет полей в исходном файле</td></tr>
              <tr><td>SpaceMatrix</td><td>Не включён: кластеризация на этих данных даёт почти все кварталы в одном классе</td></tr>
            </tbody>
          </table>
        </div>
      </div>
      <div class="card">
        <h3>Ограничения</h3>
        <ul class="clean">
          <li><b>Население — оценка.</b> Оно получено агрегированием жилплощади и норматива, а не официальной статистикой.</li>
          <li><b>OSM неполон.</b> Пропущенные здания и объекты могут занижать плотность и доступность.</li>
          <li><b>site_area в EPSG:3857.</b> Исходные FSI/GSI занижены; в выводах используем пересчитанные значения.</li>
          <li><b>Морфотип чувствителен к площади.</b> При пересчёте часть категорий меняется; это показано в таблице.</li>
          <li><b>Нет школ и обеспеченности.</b> В файле нет соответствующих полей, поэтому они не рассчитывались.</li>
          <li><b>Выборка — __N_BLOCKS__ кварталов после очистки.</b> В исходном файле было __N_SOURCE__; это не вся территория муниципального образования.</li>
        </ul>
      </div>
    </div>

    <details>
      <summary>Почему не включён SpaceMatrix</summary>
      <p>В BlocksNet SpaceMatrix строится через кластеризацию кварталов по FSI, GSI и L. На этих данных KMeans даёт почти все кварталы в одном классе «low-density», поэтому график не различает районы и создаёт ложное ощущение типологии. Мы сохранили FSI/GSI/L и морфотипы Strelka — они информативнее для этой выборки.</p>
    </details>
    <details>
      <summary>Что именно изменено в данных</summary>
      <p>Исправлены невалидные геометрии, площади пересчитаны в UTM 36N, категории землепользования нормализованы, FSI/GSI пересчитаны на метрическую площадь. Исходные значения сохранены в колонках <code>fsi</code>, <code>gsi</code>, <code>site_area</code>; производные — в <code>fsi_calc</code>, <code>gsi_calc</code>, <code>area_m2</code>.</p>
    </details>
  </section>
</main>

<footer class="footer">
  <div class="wrap">
    <p><b>__CITY__: индикаторы кварталов.</b> Страница собрана автоматически из ноутбука <code>examples/gatchina_dashboard.ipynb</code>. Данные: <code>examples/data/blocks_agg_with_indicators.geojson</code>.</p>
    <p>Обновлено: __UPDATED__. Библиотеки карты и графиков встроены в файл; подложка OpenStreetMap требует интернет.</p>
  </div>
</footer>

<script id="blocks-data" type="application/json">__BLOCKS_JSON__</script>
<script id="chart-data" type="application/json">__CHART_DATA_JSON__</script>
<script id="map-metrics" type="application/json">__MAP_METRICS_JSON__</script>
<script>__LEAFLET_JS__</script>
<script>__CHART_JS__</script>
<script>__APP_JS__</script>
</body>
</html>
"""


In [11]:
APP_JS = r"""
const BLOCKS = JSON.parse(document.getElementById('blocks-data').textContent);
const CHART_DATA = JSON.parse(document.getElementById('chart-data').textContent);
const MAP_METRICS = JSON.parse(document.getElementById('map-metrics').textContent);

const PALETTES = {
  blue: ['#eff3ff', '#bdd7e7', '#6baed6', '#3182bd', '#08519c'],
  access: ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027'],
};
const state = {
  metric: MAP_METRICS[0],
  breaks: [],
  features: [],
  opacity: 0.82,
};
let map;
let osmLayer;
let geojsonLayer;

function escapeHtml(text) {
  return String(text).replace(/[&<>"']/g, s => ({
    '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;'
  }[s]));
}

function formatNumber(value, digits = 2) {
  if (value === null || value === undefined || Number.isNaN(Number(value))) return '—';
  return new Intl.NumberFormat('ru-RU', { maximumFractionDigits: digits }).format(Number(value));
}

function formatMapValue(metric, value) {
  if (value === null || value === undefined || value === '') return 'нет данных';
  if (metric.type === 'categorical') return String(value);
  const number = Number(value);
  if (!Number.isFinite(number)) return 'нет данных';
  const digits = metric.decimals === undefined ? 2 : metric.decimals;
  return formatNumber(number, digits) + (metric.unit ? ' ' + metric.unit : '');
}

function getMetricValue(feature, metric) {
  const value = feature.properties[metric.key];
  return (value === null || value === undefined || value === '') ? null : value;
}

function quantileBreaks(values, classes) {
  const numbers = values
    .filter(v => v !== null && v !== undefined && Number.isFinite(Number(v)))
    .map(Number)
    .sort((a, b) => a - b);
  if (numbers.length === 0) return [];
  const breaks = [];
  for (let i = 1; i < classes; i++) {
    const index = Math.floor(i * numbers.length / classes);
    if (index > 0 && index < numbers.length) breaks.push(numbers[index]);
  }
  return [...new Set(breaks)].sort((a, b) => a - b);
}

function classIndex(value, breaks) {
  let index = 0;
  while (index < breaks.length && Number(value) > breaks[index]) index++;
  return index;
}

function styleFeature(feature) {
  const metric = state.metric;
  const value = getMetricValue(feature, metric);
  if (metric.type === 'categorical') {
    const color = (metric.colors && metric.colors[String(value)]) || '#9e9e9e';
    return { fillColor: color, fillOpacity: state.opacity, color: '#ffffff', weight: 0.7, opacity: 1 };
  }
  if (value === null || !Number.isFinite(Number(value))) {
    return { fillColor: '#d9d9d9', fillOpacity: Math.max(0.25, state.opacity - 0.2), color: '#ffffff', weight: 0.7, opacity: 1 };
  }
  const palette = PALETTES[metric.palette] || PALETTES.blue;
  const index = classIndex(value, state.breaks);
  return {
    fillColor: palette[Math.min(index, palette.length - 1)],
    fillOpacity: state.opacity,
    color: '#ffffff',
    weight: 0.7,
    opacity: 1,
  };
}

function tooltipHtml(feature) {
  const p = feature.properties;
  return '<b>Квартал #' + escapeHtml(p.block_id) + '</b><br>' +
    escapeHtml(p.land_use_ru) + ' · ' + escapeHtml(p.morphotype_ru) + '<br>' +
    '<span class="tt-label">Население:</span> ' + formatNumber(p.population, 0) + ' чел.<br>' +
    '<span class="tt-label">FSI:</span> ' + formatNumber(p.fsi_calc, 3) +
    ' · <span class="tt-label">GSI:</span> ' + formatNumber(p.gsi_calc, 3) + '<br>' +
    '<span class="tt-label">Доступность:</span> ' + formatNumber(p.area_accessibility, 2) + ' мин';
}

function popupHtml(feature) {
  const p = feature.properties;
  const rows = [
    ['Землепользование', p.land_use_ru],
    ['Морфотип', p.morphotype_ru],
    ['Население', formatNumber(p.population, 0) + ' чел.'],
    ['Площадь', formatNumber(p.area_ha, 2) + ' га'],
    ['Плотность', formatNumber(p.population_density, 1) + ' чел./га'],
    ['FSI (файл)', formatNumber(p.fsi, 3)],
    ['FSI (пересчёт)', formatNumber(p.fsi_calc, 3)],
    ['GSI (файл)', formatNumber(p.gsi, 3)],
    ['GSI (пересчёт)', formatNumber(p.gsi_calc, 3)],
    ['MXI', formatNumber(p.mxi, 3)],
    ['L (этажность)', formatNumber(p.l, 2)],
    ['Доступность', formatNumber(p.area_accessibility, 2) + ' мин'],
    ['Жилплощадь на чел.', formatNumber(p.living_per_capita, 1) + ' м²'],
  ];
  return '<b>Квартал #' + escapeHtml(p.block_id) + '</b>' +
    '<table class="popup-table"><tbody>' +
    rows.map(([key, value]) => '<tr><td>' + escapeHtml(key) + '</td><td>' + escapeHtml(value) + '</td></tr>').join('') +
    '</tbody></table>';
}

function onEachFeature(feature, layer) {
  layer.bindTooltip(tooltipHtml(feature), { sticky: true, direction: 'top', opacity: 0.96 });
  layer.bindPopup(popupHtml(feature), { maxWidth: 360 });
  layer.on({
    mouseover: event => {
      const target = event.target;
      target.setStyle({ weight: 2.2, color: '#17212b' });
      target.bringToFront();
    },
    mouseout: event => {
      geojsonLayer.resetStyle(event.target);
    },
  });
}

function filteredFeatures() {
  const landUse = document.getElementById('filter-landuse').value;
  const morph = document.getElementById('filter-morph').value;
  return BLOCKS.features.filter(feature => {
    const p = feature.properties;
    return (landUse === 'all' || p.land_use_ru === landUse) &&
           (morph === 'all' || p.morphotype_ru === morph);
  });
}

function renderLegend(features) {
  const metric = state.metric;
  const container = document.getElementById('map-legend');
  const values = features.map(f => getMetricValue(f, metric));
  let html = '<h4>' + escapeHtml(metric.label) + '</h4>';

  if (metric.type === 'categorical') {
    const counts = {};
    features.forEach(feature => {
      const value = String(getMetricValue(feature, metric) ?? 'Нет данных');
      counts[value] = (counts[value] || 0) + 1;
    });
    const known = Object.keys(metric.colors || {});
    const extra = Object.keys(counts).filter(key => !known.includes(key)).sort();
    [...known, ...extra].forEach(name => {
      const count = counts[name] || 0;
      if (!count) return;
      const color = (metric.colors && metric.colors[name]) || '#9e9e9e';
      html += '<div class="legend-item"><span class="swatch" style="background:' + color + '"></span>' +
        escapeHtml(name) + ' <span style="color:#7a8996">(' + count + ')</span></div>';
    });
    if (!Object.keys(counts).length) html += '<div class="legend-note">Нет видимых кварталов.</div>';
    container.innerHTML = html;
    return;
  }

  const numbers = values.filter(v => v !== null && v !== undefined && Number.isFinite(Number(v))).map(Number);
  if (!numbers.length) {
    container.innerHTML = html + '<div class="legend-note">Нет числовых данных для выбранных кварталов.</div>';
    return;
  }
  const low = Math.min(...numbers);
  const high = Math.max(...numbers);
  const palette = PALETTES[metric.palette] || PALETTES.blue;
  const ranges = [];
  let start = low;
  state.breaks.forEach(br => {
    if (br > start) {
      ranges.push([start, br]);
      start = br;
    }
  });
  ranges.push([start, high]);
  if (ranges.length === 1 && low === high) {
    ranges[0] = [low, high];
  }
  ranges.forEach(([from, to], index) => {
    const color = palette[Math.min(index, palette.length - 1)];
    const text = from === to
      ? formatMapValue(metric, from)
      : formatMapValue(metric, from) + ' – ' + formatMapValue(metric, to);
    html += '<div class="legend-item"><span class="swatch" style="background:' + color + '"></span>' + escapeHtml(text) + '</div>';
  });
  if (metric.palette === 'access') {
    html += '<div class="legend-note">Зелёный — лучше, красный — хуже.</div>';
  } else {
    html += '<div class="legend-note">Светлый — меньше, тёмный — больше.</div>';
  }
  container.innerHTML = html;
}

function renderMap(resetView = false) {
  state.features = filteredFeatures();
  const values = state.features.map(f => getMetricValue(f, state.metric));
  state.breaks = state.metric.type === 'numeric' ? quantileBreaks(values, 5) : [];
  geojsonLayer.clearLayers();
  geojsonLayer.addData({ type: 'FeatureCollection', features: state.features });
  geojsonLayer.setStyle(styleFeature);
  renderLegend(state.features);
  document.getElementById('map-count').textContent =
    'Показано кварталов: ' + state.features.length + ' из ' + BLOCKS.features.length;
  if (resetView) {
    const bounds = geojsonLayer.getBounds();
    if (bounds.isValid()) map.fitBounds(bounds, { padding: [20, 20] });
  }
}

function fillSelect(id, values, allLabel) {
  const select = document.getElementById(id);
  select.innerHTML = '';
  const all = document.createElement('option');
  all.value = 'all';
  all.textContent = allLabel;
  select.appendChild(all);
  values.forEach(value => {
    const option = document.createElement('option');
    option.value = value;
    option.textContent = value;
    select.appendChild(option);
  });
}

function initMap() {
  map = L.map('map', { zoomControl: true, scrollWheelZoom: true });
  osmLayer = L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
    maxZoom: 19,
    attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>',
  }).addTo(map);
  L.control.scale({ imperial: false }).addTo(map);
  const initialLayer = L.geoJSON(BLOCKS);
  const initialBounds = initialLayer.getBounds();
  if (initialBounds.isValid()) map.fitBounds(initialBounds, { padding: [20, 20] });
  geojsonLayer = L.geoJSON(null, { style: styleFeature, onEachFeature: onEachFeature }).addTo(map);
}

function initControls() {
  const metricSelect = document.getElementById('metric-select');
  MAP_METRICS.forEach(metric => {
    const option = document.createElement('option');
    option.value = metric.key;
    option.textContent = metric.label;
    metricSelect.appendChild(option);
  });
  metricSelect.value = MAP_METRICS[0].key;

  const landUseValues = [...new Set(BLOCKS.features.map(f => f.properties.land_use_ru))].sort((a, b) => a.localeCompare(b, 'ru'));
  const morphValues = [...new Set(BLOCKS.features.map(f => f.properties.morphotype_ru))].sort((a, b) => a.localeCompare(b, 'ru'));
  fillSelect('filter-landuse', landUseValues, 'Все землепользования');
  fillSelect('filter-morph', morphValues, 'Все морфотипы');

  metricSelect.addEventListener('change', event => {
    state.metric = MAP_METRICS.find(metric => metric.key === event.target.value) || MAP_METRICS[0];
    renderMap(false);
  });
  document.getElementById('filter-landuse').addEventListener('change', () => renderMap(false));
  document.getElementById('filter-morph').addEventListener('change', () => renderMap(false));
  document.getElementById('opacity').addEventListener('input', event => {
    state.opacity = Number(event.target.value) / 100;
    geojsonLayer.setStyle(styleFeature);
  });
  document.getElementById('basemap').addEventListener('change', event => {
    if (event.target.checked) {
      osmLayer.addTo(map);
    } else {
      map.removeLayer(osmLayer);
    }
  });
  document.getElementById('reset-map').addEventListener('click', () => {
    metricSelect.value = MAP_METRICS[0].key;
    state.metric = MAP_METRICS[0];
    document.getElementById('filter-landuse').value = 'all';
    document.getElementById('filter-morph').value = 'all';
    document.getElementById('opacity').value = '82';
    state.opacity = 0.82;
    document.getElementById('basemap').checked = true;
    if (!map.hasLayer(osmLayer)) osmLayer.addTo(map);
    renderMap(true);
  });

  renderMap(true);
}

function baseBarOptions(unit, horizontal) {
  return {
    indexAxis: horizontal ? 'y' : 'x',
    responsive: true,
    maintainAspectRatio: false,
    animation: false,
    plugins: {
      legend: { display: false },
      tooltip: {
        callbacks: {
          label: ctx => {
            const value = ctx.parsed.x !== undefined && horizontal ? ctx.parsed.x : ctx.parsed.y;
            return formatNumber(value, 2) + (unit ? ' ' + unit : '');
          },
        },
      },
    },
    scales: horizontal ? {
      x: { beginAtZero: true, grid: { color: '#e9eef3' } },
      y: { grid: { display: false }, ticks: { font: { size: 11 } } },
    } : {
      x: { grid: { display: false }, ticks: { maxRotation: 55, minRotation: 0, font: { size: 10 } } },
      y: { beginAtZero: true, grid: { color: '#e9eef3' } },
    },
  };
}

function barChart(id, labels, values, label, color, unit, horizontal = false) {
  new Chart(document.getElementById(id).getContext('2d'), {
    type: 'bar',
    data: {
      labels: labels,
      datasets: [{ label: label, data: values, backgroundColor: color, borderRadius: 4 }],
    },
    options: baseBarOptions(unit, horizontal),
  });
}

function initCharts() {
  Chart.defaults.font.family = '-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif';
  Chart.defaults.color = '#5b6b7c';

  barChart('chart-landuse-area', CHART_DATA.landuse.labels, CHART_DATA.landuse.area_ha, 'Площадь, га', CHART_DATA.landuse.colors, 'га', true);
  barChart('chart-landuse-pop', CHART_DATA.landuse.labels, CHART_DATA.landuse.population, 'Население, чел.', CHART_DATA.landuse.colors, 'чел.', true);
  barChart('chart-access', CHART_DATA.access.labels, CHART_DATA.access.counts, 'Кварталов', '#2f6f8f', 'кварталов', false);
  barChart('chart-fsi-hist', CHART_DATA.fsi_hist.labels, CHART_DATA.fsi_hist.counts, 'Кварталов', '#2f6f8f', 'кварталов', false);
  barChart('chart-gsi-hist', CHART_DATA.gsi_hist.labels, CHART_DATA.gsi_hist.counts, 'Кварталов', '#76b7b2', 'кварталов', false);
  barChart('chart-morph-count', CHART_DATA.morph.labels, CHART_DATA.morph.counts, 'Кварталов', CHART_DATA.morph.colors, 'кварталов', false);
  barChart('chart-morph-pop', CHART_DATA.morph.labels, CHART_DATA.morph.population, 'Население, чел.', CHART_DATA.morph.colors, 'чел.', false);
  barChart('chart-top-pop', CHART_DATA.top_pop.labels, CHART_DATA.top_pop.values, 'Население, чел.', '#2f6f8f', 'чел.', true);

  new Chart(document.getElementById('chart-fsi-compare').getContext('2d'), {
    type: 'bar',
    data: {
      labels: CHART_DATA.fsi_compare.labels,
      datasets: [
        { label: 'Как в файле', data: CHART_DATA.fsi_compare.source, backgroundColor: '#9bb7c9', borderRadius: 4 },
        { label: 'Пересчитанный', data: CHART_DATA.fsi_compare.calc, backgroundColor: '#2f6f8f', borderRadius: 4 },
      ],
    },
    options: {
      responsive: true, maintainAspectRatio: false, animation: false,
      plugins: {
        legend: { display: true, position: 'bottom' },
        tooltip: { callbacks: { label: ctx => ctx.dataset.label + ': ' + formatNumber(ctx.parsed.y, 4) } },
      },
      scales: { y: { beginAtZero: true, grid: { color: '#e9eef3' } }, x: { grid: { display: false } } },
    },
  });

  new Chart(document.getElementById('chart-morph-compare').getContext('2d'), {
    type: 'bar',
    data: {
      labels: CHART_DATA.morph.labels,
      datasets: [
        { label: 'Как в файле', data: CHART_DATA.morph.source_counts, backgroundColor: '#9bb7c9', borderRadius: 4 },
        { label: 'После пересчёта', data: CHART_DATA.morph.recalc_counts, backgroundColor: '#d99b3d', borderRadius: 4 },
      ],
    },
    options: {
      responsive: true, maintainAspectRatio: false, animation: false,
      plugins: {
        legend: { display: true, position: 'bottom' },
        tooltip: { callbacks: { label: ctx => ctx.dataset.label + ': ' + formatNumber(ctx.parsed.y, 0) + ' кв.' } },
      },
      scales: { y: { beginAtZero: true, grid: { color: '#e9eef3' } }, x: { grid: { display: false }, ticks: { maxRotation: 55, minRotation: 0, font: { size: 10 } } } },
    },
  });

  new Chart(document.getElementById('chart-scatter').getContext('2d'), {
    type: 'scatter',
    data: {
      datasets: CHART_DATA.scatter.datasets.map(dataset => Object.assign({}, dataset, {
        pointRadius: 4,
        pointHoverRadius: 7,
        borderWidth: 0,
      })),
    },
    options: {
      responsive: true, maintainAspectRatio: false, animation: false,
      plugins: {
        legend: { display: true, position: 'bottom', labels: { boxWidth: 10, font: { size: 10 } } },
        tooltip: {
          callbacks: {
            label: ctx => {
              const point = ctx.raw;
              return '#' + point.id + ': FSI ' + formatNumber(point.x, 3) + ', L ' + formatNumber(point.y, 2) + ', население ' + formatNumber(point.population, 0);
            },
          },
        },
      },
      scales: {
        x: { title: { display: true, text: 'FSI (пересчитанный)' }, grid: { color: '#e9eef3' } },
        y: { title: { display: true, text: 'Средняя этажность L' }, beginAtZero: true, grid: { color: '#e9eef3' } },
      },
    },
  });
}

function downloadCSV() {
  const columns = [
    'block_id', 'land_use_ru', 'morphotype_ru', 'morphotype_recalc', 'population',
    'population_density', 'area_ha', 'site_area', 'footprint_area', 'build_floor_area',
    'living_area', 'non_living_area', 'fsi', 'fsi_calc', 'gsi', 'gsi_calc',
    'mxi', 'l', 'area_accessibility', 'access_class', 'living_per_capita', 'floor_per_capita',
  ];
  const lines = [columns.join(';')];
  BLOCKS.features.forEach(feature => {
    const p = feature.properties;
    lines.push(columns.map(column => {
      const value = p[column];
      if (value === null || value === undefined) return '';
      return String(value).replace(/;/g, ',').replace(/\r?\n/g, ' ');
    }).join(';'));
  });
  const blob = new Blob(['\ufeff' + lines.join('\n')], { type: 'text/csv;charset=utf-8' });
  const url = URL.createObjectURL(blob);
  const link = document.createElement('a');
  link.href = url;
  link.download = 'gatchina_indicators.csv';
  document.body.appendChild(link);
  link.click();
  document.body.removeChild(link);
  setTimeout(() => URL.revokeObjectURL(url), 1500);
}

document.addEventListener('DOMContentLoaded', () => {
  initMap();
  initControls();
  initCharts();
});
"""


In [12]:
VENDOR_URLS = {
    "leaflet.css": "https://unpkg.com/leaflet@1.9.4/dist/leaflet.css",
    "leaflet.js": "https://unpkg.com/leaflet@1.9.4/dist/leaflet.js",
    "chart.umd.min.js": "https://cdn.jsdelivr.net/npm/chart.js@4.4.7/dist/chart.umd.min.js",
}


def ensure_vendor():
    VENDOR.mkdir(parents=True, exist_ok=True)
    for name, url in VENDOR_URLS.items():
        path = VENDOR / name
        if not path.exists() or path.stat().st_size < 1000:
            print("Скачиваю библиотеку:", name)
            urllib.request.urlretrieve(url, path)
    return {name: (VENDOR / name).read_text(encoding="utf-8") for name in VENDOR_URLS}


vendor = ensure_vendor()
leaflet_css = vendor["leaflet.css"]
leaflet_js = vendor["leaflet.js"]
chart_js = vendor["chart.umd.min.js"]

LAND_USE_COLORS_MAP = LAND_USE_COLORS
MORPH_COLORS_MAP = MORPH_COLORS

MAP_METRICS = [
    {"key": "population", "label": "Население", "unit": "чел.", "type": "numeric", "palette": "blue", "decimals": 0},
    {"key": "population_density", "label": "Плотность населения", "unit": "чел./га", "type": "numeric", "palette": "blue", "decimals": 1},
    {"key": "area_ha", "label": "Площадь квартала", "unit": "га", "type": "numeric", "palette": "blue", "decimals": 2},
    {"key": "fsi", "label": "FSI (как в файле)", "unit": "", "type": "numeric", "palette": "blue", "decimals": 3},
    {"key": "fsi_calc", "label": "FSI (пересчитанный)", "unit": "", "type": "numeric", "palette": "blue", "decimals": 3},
    {"key": "gsi", "label": "GSI (как в файле)", "unit": "", "type": "numeric", "palette": "blue", "decimals": 3},
    {"key": "gsi_calc", "label": "GSI (пересчитанный)", "unit": "", "type": "numeric", "palette": "blue", "decimals": 3},
    {"key": "mxi", "label": "MXI (доля жилой площади)", "unit": "", "type": "numeric", "palette": "blue", "decimals": 3},
    {"key": "l", "label": "Средняя этажность L", "unit": "эт.", "type": "numeric", "palette": "blue", "decimals": 2},
    {"key": "area_accessibility", "label": "Доступность территории", "unit": "мин", "type": "numeric", "palette": "access", "decimals": 2},
    {"key": "land_use_ru", "label": "Землепользование", "type": "categorical", "colors": LAND_USE_COLORS_MAP},
    {"key": "morphotype_ru", "label": "Морфотип (из файла)", "type": "categorical", "colors": MORPH_COLORS_MAP},
    {"key": "morphotype_recalc", "label": "Морфотип (пересчитанный)", "type": "categorical", "colors": MORPH_COLORS_MAP},
]

map_props = [
    "block_id", "land_use_ru", "morphotype_ru", "morphotype_recalc",
    "population", "population_density", "area_m2", "area_ha", "site_area",
    "footprint_area", "build_floor_area", "living_area", "non_living_area",
    "fsi", "fsi_calc", "gsi", "gsi_calc", "mxi", "l", "area_accessibility",
    "access_class", "living_per_capita", "floor_per_capita", "non_living_share",
    "land_use_share_total",
]
map_features = []
for _, row in gdf.to_crs(4326).iterrows():
    properties = {column: json_safe(row[column]) for column in map_props}
    map_features.append(
        {
            "type": "Feature",
            "properties": properties,
            "geometry": mapping(row.geometry),
        }
    )
blocks_geojson = {"type": "FeatureCollection", "features": map_features}

callout = (
    "<b>Важно про площади.</b> В исходном файле <code>site_area</code> посчитана в EPSG:3857, "
    f"а не в метрической проекции. Она в <b>{site_factor:.2f} раза</b> больше реальной площади, "
    "поэтому исходные FSI/GSI занижены. В карточках, карте и выводах используем пересчитанные "
    "<code>fsi_calc</code> и <code>gsi_calc</code>; исходные значения оставлены для сверки."
)

replacements = {
    "__LEAFLET_CSS__": leaflet_css,
    "__LEAFLET_JS__": leaflet_js,
    "__CHART_JS__": chart_js,
    "__APP_JS__": APP_JS,
    "__KPI_CARDS__": kpi_cards,
    "__CONCLUSIONS__": conclusions_html,
    "__CALLOUT__": callout,
    "__METRIC_TABLE__": metric_table,
    "__SUMMARY_TABLE__": summary_table,
    "__QUALITY_TABLE__": quality_table(validation_df),
    "__LAND_USE_TABLE__": land_use_table,
    "__MORPH_TABLE__": morph_table,
    "__TOP_POP_TABLE__": top_pop_table,
    "__TOP_FSI_TABLE__": top_fsi_table,
    "__ACCESS_TABLE__": access_table,
    "__MORPH_CHANGES_TABLE__": morph_changes_table,
    "__BLOCKS_JSON__": json.dumps(blocks_geojson, ensure_ascii=False, allow_nan=False, separators=(",", ":")).replace("</", "<\\/"),
    "__CHART_DATA_JSON__": json.dumps(to_jsonable(chart_data), ensure_ascii=False, allow_nan=False, separators=(",", ":")).replace("</", "<\\/"),
    "__MAP_METRICS_JSON__": json.dumps(to_jsonable(MAP_METRICS), ensure_ascii=False, allow_nan=False, separators=(",", ":")).replace("</", "<\\/"),
    "__UPDATED__": date.today().isoformat(),
    "__CITY__": html.escape(CITY),
    "__REGION__": html.escape(REGION),
    "__SOURCE_NOTE__": html.escape(SOURCE_NOTE),
    "__N_BLOCKS__": str(len(gdf)),
    "__N_SOURCE__": str(blocks_before_cleaning),
}

HTML_TEMPLATE = HTML_TEMPLATE_TOP + HTML_TEMPLATE_BOTTOM
html_text = HTML_TEMPLATE
for placeholder, value in replacements.items():
    html_text = html_text.replace(placeholder, str(value))

unresolved = sorted(set(re.findall(r"__[A-Z0-9_]+__", html_text)))
if unresolved:
    raise RuntimeError("Не заменены плейсхолдеры: " + ", ".join(unresolved))

HTML_PATH.write_text(html_text, encoding="utf-8")

print("HTML собран:", HTML_PATH)
print("Размер, МБ:", round(HTML_PATH.stat().st_size / 1e6, 2))
print("Кварталов в JSON:", len(blocks_geojson["features"]))
print("Метрик на карте:", len(MAP_METRICS))


HTML собран: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\gatchina_dashboard.html
Размер, МБ: 1.05
Кварталов в JSON: 349
Метрик на карте: 13


## Финальная проверка

Проверяем не только наличие файла, но и то, что HTML действительно содержит валидный JSON с кварталами, все нужные секции и что итоговые суммы совпадают с расчётом.


In [13]:
required_files = [
    HTML_PATH,
    OUT_DATA / "blocks_clean.geojson",
    OUT_DATA / "indicators.csv",
    OUT_DATA / "summary.json",
    OUT_DATA / "summary.csv",
    OUT_DATA / "land_use_summary.csv",
    OUT_DATA / "morphotype_summary.csv",
    OUT_DATA / "validation_report.csv",
    OUT_DATA / "chart_data.json",
]
for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Не создан обязательный файл: {path}")
    if path.stat().st_size == 0:
        raise RuntimeError(f"Пустой файл: {path}")
    print(f"OK  {path.name:28s} {path.stat().st_size:>12,} байт".replace(",", " "))

html_text = HTML_PATH.read_text(encoding="utf-8")


def extract_json(script_id):
    match = re.search(rf'<script id="{script_id}" type="application/json">(.*?)</script>', html_text, re.S)
    if not match:
        raise RuntimeError(f"В HTML нет JSON-блока {script_id}")
    return json.loads(match.group(1))


blocks_check = extract_json("blocks-data")
chart_check = extract_json("chart-data")
metrics_check = extract_json("map-metrics")

if len(blocks_check["features"]) != len(gdf):
    raise RuntimeError("Число кварталов в HTML не совпадает с расчётом")
if len(metrics_check) < 10:
    raise RuntimeError("Слишком мало метрик на карте")
if len(chart_check["landuse"]["labels"]) == 0:
    raise RuntimeError("Нет данных для графика землепользования")
if "<div id=\"map\"></div>" not in html_text:
    raise RuntimeError("В HTML нет контейнера карты")
if html_text.count("<canvas") < 10:
    raise RuntimeError("В HTML меньше 10 графиков")
if "__BLOCKS_JSON__" in html_text or "__APP_JS__" in html_text:
    raise RuntimeError("В HTML остались незаменённые плейсхолдеры")

pop_html = sum(float(f["properties"]["population"] or 0) for f in blocks_check["features"])
if abs(pop_html - float(gdf["population"].sum())) > 1:
    raise RuntimeError("Сумма населения в HTML не совпадает с расчётом")

area_html = sum(float(f["properties"]["area_m2"] or 0) for f in blocks_check["features"])
if abs(area_html - float(gdf["area_m2"].sum())) > 1:
    raise RuntimeError("Сумма площадей в HTML не совпадает с расчётом")

print("\nПроверка HTML: успешно")
print("Кварталов в HTML:", len(blocks_check["features"]))
print("Население в HTML:", fmt_int(pop_html), "чел.")
print("Площадь в HTML:", fmt_num(area_html / 1e6, 2), "км²")
print("Графиков <canvas>:", html_text.count("<canvas"))
print("Метрик на карте:", len(metrics_check))
print("Размер HTML:", round(len(html_text.encode("utf-8")) / 1e6, 2), "МБ")


OK  gatchina_dashboard.html         1 045 774 байт
OK  blocks_clean.geojson              789 642 байт
OK  indicators.csv                    220 992 байт
OK  summary.json                        3 031 байт
OK  summary.csv                         1 270 байт
OK  land_use_summary.csv                1 247 байт
OK  morphotype_summary.csv              1 121 байт
OK  validation_report.csv               3 963 байт
OK  chart_data.json                    50 768 байт



Проверка HTML: успешно
Кварталов в HTML: 349
Население в HTML: 91 521 чел.
Площадь в HTML: 33,32 км²
Графиков <canvas>: 11
Метрик на карте: 13
Размер HTML: 1.04 МБ


## Выводы

Ниже — те же выводы, что попали на HTML-страницу. Они автоматически собираются из расчётов, поэтому при повторном запуске на других данных текст обновится.


In [14]:
for item in conclusions:
    print("•", item)

print("\nГотово. Откройте страницу:")
print(HTML_PATH)


• В выборке 349 кварталов общей площадью 33,3 км²; оценка населения — 91 521 чел.
• По площади лидирует категория «Жилая»: 11,7 км², это 35,1% территории.
• Больше всего жителей — в морфотипе «Среднеэтажная»: 54 702 чел.
• Средняя доступность территории — 10,1 мин; 215 кварталов (61,6%) имеют доступность до 10 минут, 13 (3,7%) — более 15 минут.
• Самый населённый квартал — #329: 11 087 чел., 58,6 га, FSI 0,863, GSI 0,132.
• Исходная site_area посчитана в EPSG:3857 и в 3,88 раза больше метрической площади. Поэтому исходные FSI/GSI занижены; в выводах используем пересчитанные значения, исходные показываем для сверки.
• При пересчёте Strelka по исправленному FSI у 135 кварталов меняется категория. Это не ошибка данных, а чувствительность морфотипа к площади: классификация пересчитывает плотность на другой знаменатель.
• После очистки осталось 349 из 351 кварталов: 2 геометрии схлопнулись в линии нулевой площади и не могут участвовать в плотностных расчётах.
• Ограничения: население — оцен

## Повторный запуск

Полный цикл повторяется одной командой:

```powershell
cd "C:\Users\pc\Desktop\р\python-package-boilerplate"
.\.venv\Scripts\jupyter.exe nbconvert --to notebook --execute --inplace examples\gatchina_dashboard.ipynb
```

Проверить, что всё на месте, можно так:

```powershell
Get-Item examples\gatchina_dashboard.html
Get-ChildItem examples\data\gatchina
```

Если меняется исходный GeoJSON, положите его на место `examples/data/blocks_agg_with_indicators.geojson` и снова выполните команду выше. Ноутбук сам пересчитает площади, FSI/GSI, морфотипы, графики, таблицы и выводы.

**Важно:** не редактируйте HTML вручную — при следующем запуске он перезапишется. Правки методики делайте в ноутбуке.


In [15]:
readme_text = f"""# Гатчина: индикаторы кварталов

## Что это

Одностраничный дашборд по {len(gdf)} кварталам Гатчины (в исходном файле было
{blocks_before_cleaning}; две геометрии нулевой площади отброшены), собранный из файла
`examples/data/blocks_agg_with_indicators.geojson`.

Страница: `examples/gatchina_dashboard.html`

Ноутбук: `examples/gatchina_dashboard.ipynb`

## Что внутри

- интерактивная карта с переключением метрик и фильтрами;
- KPI-карточки;
- графики по землепользованию, доступности, FSI/GSI, морфотипам и топ-10;
- сводные таблицы, топ-10 и проверки качества;
- объяснение показателей, методики и ограничений;
- автоматические выводы.

## Как запустить

```powershell
cd "C:\\Users\\pc\\Desktop\\р\\python-package-boilerplate"
.\\.venv\\Scripts\\jupyter.exe nbconvert --to notebook --execute --inplace examples\\gatchina_dashboard.ipynb
```

Или откройте ноутбук в Jupyter Lab и выполните **Kernel → Restart & Run All**.

## Результаты

- `examples/gatchina_dashboard.html` — готовая страница;
- `examples/data/gatchina/blocks_clean.geojson` — очищенные кварталы;
- `examples/data/gatchina/indicators.csv` — показатели по кварталам;
- `examples/data/gatchina/summary.json` и `summary.csv` — городские итоги;
- `examples/data/gatchina/land_use_summary.csv`;
- `examples/data/gatchina/morphotype_summary.csv`;
- `examples/data/gatchina/validation_report.csv`;
- `examples/data/gatchina/chart_data.json`.

## Важное ограничение

В исходном файле `site_area` посчитана в EPSG:3857 и примерно в {ru_num(site_factor, 2)} раза больше
метрической площади. Поэтому FSI/GSI пересчитаны на `area_m2` в UTM 36N; исходные значения
сохранены для сверки. Население — оценка, а не официальная статистика.
"""

(EXAMPLES / "gatchina_README.md").write_text(readme_text, encoding="utf-8")
print("README сохранён:", EXAMPLES / "gatchina_README.md")


README сохранён: C:\Users\pc\Desktop\р\python-package-boilerplate\examples\gatchina_README.md
